# NEXUS Alpha: the end-to-end research pipeline in one notebook (v5)

**Neural Exchange for Unified Signal Intelligence.** A multimodal research system for US stocks. It reads weekly, daily
and hourly charts, technical indicators, company news and SEC filings, and forecasts six horizons (1, 2, 5, 10, 20 and
21 sessions) as probabilities of UP, SIDEWAYS or DOWN. A separate, deterministic risk engine turns a forecast into
LONG, SHORT or NO TRADE, with entry, stop, targets and size.

This notebook holds the code of every step, with a short explanation above each one:

| # | Step | What you will see |
|---|---|---|
| 1 | Setup and governance | GPU check, inputs on Google Drive, checksums, the sealed-period guard |
| 2 | Data | 236,520 stock-dates of 500 US stocks, 2001-2022; coverage by year and block |
| 3 | Labels | UP / SIDEWAYS / DOWN from volatility-scaled returns, recomputed and checked |
| 4 | Splits | train, early stopping, validation, calibration and test blocks, purged; the purge is re-verified |
| 5 | Features | 80 indicator features (weekly, daily, hourly); the formulas re-implemented and checked against the stored values |
| 6 | Baselines | class prior; multinomial logistic regression with a **global-minimum certificate** |
| 7 | LightGBM | gradient boosting per horizon, **training and validation loss curves**, feature importance |
| 8 | Fusion neural network (GPU) | charts + indicators + news + SEC, gated fusion, 3 seeds, **loss curves** |
| 8A | Language branch (GPU) | ChronoBERT, a transformer with a 2010 text cutoff, reads 68,853 SEC filings; parity with the documented vectors |
| 8B | SEC price-impact expert | the SEC study's own labels turned into a model: does a filing move the price? |
| 8C | Vision + NLP fusion (E28) | one network over charts, indicators, news and filings; the pre-registered keep test, placebo, gates |
| 9 | Optimisation | convexity, seeds, the loss landscape: what "reaching the global minimum" means here |
| 10 | Fine-tuning (GPU) | ImageNet ConvNeXt-Tiny fine-tuned on rendered chart images, **loss curves** |
| 11 | Calibration | temperature scaling, reliability diagrams, ECE |
| 12 | Evaluation | log loss, Brier, accuracy, balanced accuracy, macro-F1, AUROC, **confusion matrices**, ROC curves |
| 13 | Statistics | month-block bootstrap confidence intervals, placebo, Holm correction, rank IC |
| 13A | Modality scorecard (GPU) | **vision vs NLP vs indicators vs combinations**: every metric for each input on its own and together, the quantile ranges as stop distances, and what each input adds |
| 13B | Fundamentals (CPU) | the published scores (Beneish M, Montier C, gross profitability, magic formula, QMJ-style quality) from SEC XBRL, and the pre-registered **E30** test of whether any of them ranks returns |
| 14 | Comparison | every model side by side, with the frozen models' saved predictions and the reproduction check |
| 15 | Risk engine | entry, stop, targets and size with the reasons, for real setups |
| 16 | Backtest | equity, drawdown and Sharpe intervals of the documented backtest |
| 16A | Entry, stop-loss, exit | one fixed bracket rule (next-open entry, 1.5 ATR stop, 3 ATR target, time exit) on every modality's forecasts: win rate, expectancy in R, profit factor, MFE/MAE, against random books |
| 17 | Knowledge base (RAG) | BM25, dense and hybrid retrieval over the books and rulebooks, with page citations |
| 18 | Forward test | the public, hash-chained forward test, verified from GitHub, now with the E29b news-on log and the v1.2 refit (E31) |
| 19 | Conclusions | what works, what does not, and what comes next |
| 20 | Full report | every key table of the run written to Google Drive as one Markdown and one JSON file |

**Honest summary before you start.** Volatility regime (sideways or not) is predictable; the cross-sectional ranking of
stocks has a small positive rank IC on validation; the direction of a single stock over 1-20 sessions is close to chance;
the multimodal blocks (charts, news, filings) add little beyond the indicators; and the documented backtests show no
edge over equal-weight buy-and-hold after costs. The notebook shows each of these with numbers you can recompute.

Research and paper trading only. Nothing here is investment advice, and no order is ever sent anywhere.

## How to run it

**On Colab Pro+ (recommended).** Runtime > Change runtime type > **A100 GPU** or **L4 GPU** (T4 also works), High-RAM
on. Then Runtime > Run all. About 50-65 minutes on an A100 (13A trains five seeds per input set) and roughly 1.5-2 times that on an L4. The GPU sections are
8, 8A, 8C and 10; without a GPU they run smaller versions (8A checks 400 filings instead of all 68,853), and the
whole notebook takes a few hours on a CPU.

**Inputs (on your Google Drive, under `MyDrive/antigravity_workspace/`):**
* `nexus_artifacts/notebook/`: the compact input set staged by `scripts/colab/stage_notebook_inputs.py` (about 1 GB; section 13B also reads `fundamentals/e30_stock_months.parquet`, 23 MB, from `scripts/colab/stage_e30_inputs.py`):
  the sample frame, chart-embedding components, news/SEC/rule features, the language inputs of sections 8A-8C (SEC
  filing text and vectors, the SEC expert's table), saved predictions of the documented runs and the knowledge base. **Keep this folder private:** the knowledge base contains book text.
* `us_stocks_ohlcv/final/1day/`: daily bars, read only up to 2022-12-30 (needed by sections 5 and 10).

The public repository (code, documented metrics and forward-test logs) is cloned in section 1.

**Settings** are in the first code cell: `QUICK = True` runs everything on small samples in a few minutes (a smoke
test); `QUICK = False` is the full run.

## 1. Setup and governance

The project's evidence rests on three rules, and the notebook enforces them in code:
1. **Point in time.** Every input is available at the decision time (09:00 New York on the session after the as-of
   close). Labels use only later prices.
2. **The sealed period stays sealed.** Nothing from 2023-01-03 onward is read; the frozen models will be scored on it
   once, by the owner, after the final freeze. The sample frame ends 2022-12-01 and bars are read with a date filter.
3. **Rules before results.** Model choices are made on the validation years (2017-2018) only; the test years
   (2020-2022) are reported, never used to choose.

In [ ]:
# ---- settings ---------------------------------------------------------------------------------------------------
QUICK = False            # True: small samples, a few minutes (smoke test). False: the full run.
SEED = 7
REPO_URL = "https://github.com/harshatangirala/nexus-alpha"

import os, sys, json, math, time, hashlib, warnings, subprocess
from pathlib import Path
warnings.filterwarnings("ignore")
IN_COLAB = "google.colab" in sys.modules
T_START = time.time()

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    WS = Path("/content/drive/MyDrive/antigravity_workspace")
    REPO_DIR = Path("/content/nexus-alpha")
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, str(REPO_DIR)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lightgbm==4.7.0", "timm==1.0.31"], check=True)
else:
    WS = Path(os.environ.get("NEXUS_WS", r"G:\My Drive\antigravity_workspace"))
    REPO_DIR = Path(os.environ.get("NEXUS_REPO", r"C:\Users\harsh\nexus-alpha"))
NB = Path(os.environ.get("NEXUS_NB_INPUTS", str(WS / "nexus_artifacts" / "notebook")))
BARS_DIR = Path(os.environ.get("NEXUS_BARS_1DAY", str(WS / "us_stocks_ohlcv" / "final" / "1day")))
print("inputs:", NB, "| bars:", BARS_DIR, "| repository:", REPO_DIR)

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F
import lightgbm as lgb
from scipy import optimize, stats
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (log_loss, roc_auc_score, accuracy_score, balanced_accuracy_score, f1_score,
                             confusion_matrix, roc_curve)
try:
    from IPython.display import display, Markdown
except ImportError:                                   # plain-Python runs
    display = print
    Markdown = str

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": .3, "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 10})
CLR = {"prior": "#9aa5b1", "logistic": "#1f77b4", "lgbm": "#2ca02c", "fusion": "#d62728", "cnn_ft": "#9467bd",
       "v1.1": "#ff7f0e", "UP": "#2ca02c", "SIDEWAYS": "#7f7f7f", "DOWN": "#d62728"}
rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    p = torch.cuda.get_device_properties(0)
    print(f"GPU: {p.name}, {p.total_memory / 1e9:.0f} GB")
else:
    print("No GPU: sections 8 and 10 run smaller CPU versions.")
print("torch", torch.__version__, "| lightgbm", lgb.__version__, "| pandas", pd.__version__, "| numpy", np.__version__)

# ---- governance guard ------------------------------------------------------------------------------------------
SEAL_DATE = pd.Timestamp("2023-01-03")                  # first sealed session; nothing on or after it is read
for v in ("NEXUS_HOLDOUT_CONFIG", "NEXUS_HOLDOUT_UNLOCK"):
    assert not os.environ.get(v), f"{v} is set: this notebook never opens the sealed period"

def sha256(p: Path) -> str:
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()

man = json.loads((NB / "manifest_notebook.json").read_text(encoding="utf-8"))
bad = [k for k, v in man["files"].items() if (NB / k).stat().st_size != v["bytes"]]
assert not bad, f"inputs differ from the staging manifest: {bad[:5]}"
check = ["samples/s500_frame.parquet", "img_pca64_convnext_tiny.npy", "predictions/rows_eval.parquet"]
for k in check:
    assert sha256(NB / k) == man["files"][k]["sha256"], f"checksum mismatch: {k}"
print(f"{len(man['files'])} staged files present; checksums of {len(check)} key files verified "
      f"(staged {man['created_utc']}).")

## 2. Data

The sample frame is TriFrame's 500-stock set: one decision date every ~10 sessions per stock, staggered, from 2001-10
to 2022-12-01, **236,520 stock-dates**. Each row holds the as-of features, the labels for six horizons and its block.
The 500 stocks are survivors in today's universe, so results carry survivorship bias (stated in
`docs/LIMITATIONS.md`); the benchmark in section 16 has the same bias.

In [ ]:
FR = pd.read_parquet(NB / "samples" / "s500_frame.parquet")
FR["asof_date"] = pd.to_datetime(FR.asof_date)
assert FR.asof_date.max() < SEAL_DATE, "a sealed row is in the frame"
H6 = ["1D", "2D", "5D", "10D", "20D", "1M"]
HN = {"1D": 1, "2D": 2, "5D": 5, "10D": 10, "20D": 20, "1M": 21}
CLASSES = ["DOWN", "SIDEWAYS", "UP"]                     # label codes 0, 1, 2
BLOCKS = ["train", "inner", "valid", "calib", "test"]
print(f"{len(FR):,} rows, {FR.ticker.nunique()} stocks, {FR.asof_date.min().date()} to {FR.asof_date.max().date()}")

cov = FR.assign(year=FR.asof_date.dt.year, block=FR.block.fillna("purged")).pivot_table(
    index="year", columns="block", values="sample_id", aggfunc="count").fillna(0)
cov = cov[[c for c in BLOCKS + ["purged"] if c in cov]]
fig, ax = plt.subplots(figsize=(11, 3.4))
bottom = np.zeros(len(cov))
for c, col in zip(cov.columns, ["#1f77b4", "#aec7e8", "#ff7f0e", "#ffbb78", "#2ca02c", "#d9d9d9"]):
    ax.bar(cov.index, cov[c], bottom=bottom, color=col, label=c)
    bottom += cov[c].to_numpy()
ax.set_title("Stock-dates per year and block")
ax.legend(ncol=6, fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.1), frameon=False); plt.show()
display(FR.block.fillna("purged").value_counts().rename("rows").to_frame().T)

## 3. Labels

For each horizon h (sessions): the return from the next session's open (the earliest executable price after the
decision time) to the close h sessions later, R_h = close[t+h] / open[t+1] - 1. The threshold scales with the stock's
own volatility: tau_h = max(0.5 * sigma_t * sqrt(h), 10 bps), sigma_t the EWMA (lambda 0.94) daily volatility at t.
**UP** if R_h > tau_h, **DOWN** if R_h < -tau_h, otherwise **SIDEWAYS**. Prices only (no dividends). The rule was fixed
before any model was trained and was not tuned for class balance.

The cell recomputes every label and threshold from the stored returns and volatility and checks they match.

In [ ]:
mism = {}
for h in H6:
    tau = np.maximum(0.5 * FR.sigma_ewma * np.sqrt(HN[h]), 0.001)
    ret = FR[f"ret_{h}"]
    y = np.where(ret > tau, 2, np.where(ret < -tau, 0, 1)).astype(float)
    y[ret.isna().to_numpy()] = np.nan
    ok = FR[f"y_{h}"].notna()
    mism[h] = {"tau max abs diff": float(np.nanmax(np.abs(tau - FR[f"tau_{h}"]))),
               "label mismatches": int((y[ok] != FR.loc[ok, f"y_{h}"]).sum()), "labelled rows": int(ok.sum())}
display(pd.DataFrame(mism).T)
assert all(v["label mismatches"] == 0 for v in mism.values()), "label rule not reproduced"

Y = np.stack([FR[f"y_{h}"].fillna(-1).to_numpy(np.int64) for h in H6], 1)       # -1 = no label (not used)
RET = np.stack([FR[f"ret_{h}"].to_numpy(np.float64) for h in H6], 1)
ZV = np.stack([FR[f"z_{h}"].to_numpy(np.float32) for h in H6], 1)              # vol-scaled return (quantile heads)
def shares(b, j):
    y = Y[(FR.block == b).to_numpy(), j]
    return pd.Series(y[y >= 0]).value_counts(normalize=True)
cnt = pd.DataFrame({(b, h): shares(b, j) for b in ["train", "valid", "test"] for j, h in enumerate(H6)}).T
cnt = cnt.rename(columns=dict(enumerate(CLASSES)))[CLASSES]
fig, axs = plt.subplots(1, 3, figsize=(12, 3), sharey=True)
for ax, b in zip(axs, ["train", "valid", "test"]):
    d = cnt.loc[b]
    left = np.zeros(len(d))
    for c in CLASSES:
        ax.barh(d.index, d[c], left=left, color=CLR[c], label=c); left += d[c].to_numpy()
    ax.set_title(f"class shares, {b}"); ax.set_xlim(0, 1)
fig.legend(*axs[0].get_legend_handles_labels(), ncol=3, fontsize=8, loc="lower center", bbox_to_anchor=(0.5, -0.06),
           frameon=False)
plt.tight_layout(); plt.show()

## 4. Splits

| Block | As-of dates | Use |
|---|---|---|
| train | 2003 to 2015 | fitting |
| inner | 2016 | early stopping only |
| valid | 2017 to 2018 | model and configuration choice |
| calib | 2019 | probability calibration and thresholds |
| test | 2020 to 2022-12-01 | reported once per frozen configuration (a historical diagnostic: TriFrame read these years) |
| sealed | from 2023-01-03 | the final evaluation, once, by the owner |

**Purging.** A row stays in a block only if its longest label window (21 sessions) plus a 5-session embargo ends before
the next block starts, so no training outcome overlaps a later block's outcomes. The cell checks it independently.

In [ ]:
masks = {b: (FR.block == b).to_numpy() for b in BLOCKS}
rep = []
for a, b in zip(BLOCKS[:-1], BLOCKS[1:]):
    last_end = pd.to_datetime(FR.loc[masks[a], "end_1M"]).max()
    first_next = FR.loc[masks[b], "asof_date"].min()
    gap = int(np.busday_count(last_end.date(), first_next.date()))
    rep.append({"from": a, "to": b, "latest label end": last_end.date(), "next block starts": first_next.date(),
                "business days between": gap, "embargo respected (>= 5)": gap >= 5})
display(pd.DataFrame(rep))
assert all(r["embargo respected (>= 5)"] for r in rep)
EVAL = masks["valid"] | masks["calib"] | masks["test"]                        # rows with saved predictions
EVI = np.flatnonzero(EVAL)

## 5. Features

Eighty numeric features per row, all causal (bar t uses bars up to t only):
* **47 indicator features** on the weekly (`w_`), daily (`d_`) and hourly (`h_`) bars: ATR as a share of price, RSI(14),
  MACD(12,26,9) and its histogram in ATR units, distance to SMA20/SMA50, Bollinger %B and width, volume ratio,
  returns, realised volatility, gaps, range, close location, distance to the 52-week high, EWMA volatility.
* **33 extra price/volume features** (`x_`): overnight and intraday returns, abnormal volume, streaks, short-term RSI,
  variance ratio, autocorrelation, and Parkinson and Garman-Klass volatility.

The formulas follow TriFrame's frozen conventions. The cell below re-implements the daily indicators from raw bars and
checks them against the stored values for three stocks.

In [ ]:
STRUCTURED = ['d_atr_pct', 'd_rsi14', 'd_macd_atr', 'd_macd_hist_atr', 'd_close_sma20', 'd_close_sma50',
              'd_sma20_sma50', 'd_bb_pctb', 'd_bb_width', 'd_log_vol_ratio', 'd_ret1', 'd_ret5', 'd_ret21', 'd_ret63',
              'd_rv5', 'd_rv20', 'd_gap', 'd_range_atr', 'd_close_loc', 'd_dist_high252', 'd_sigma_ewma',
              'w_atr_pct', 'w_rsi14', 'w_macd_atr', 'w_macd_hist_atr', 'w_close_sma20', 'w_close_sma50',
              'w_sma20_sma50', 'w_bb_pctb', 'w_bb_width', 'w_log_vol_ratio', 'w_ret1', 'w_ret4', 'w_wtd_ret',
              'h_atr_pct', 'h_rsi14', 'h_macd_atr', 'h_macd_hist_atr', 'h_close_sma20', 'h_close_sma50',
              'h_sma20_sma50', 'h_bb_pctb', 'h_bb_width', 'h_log_vol_ratio', 'h_ret7', 'h_ret35',
              'h_last_session_close_loc']
X33 = [c for c in FR.columns if c.startswith("x_")]
NUM = STRUCTURED + X33
print(len(STRUCTURED), "indicator features +", len(X33), "extra =", len(NUM))

def read_daily(ticker):
    # pyarrow pushes the date filter down: no sealed bar is ever loaded into memory
    import pyarrow.parquet as pq
    from datetime import datetime
    t = pq.read_table(BARS_DIR / f"{ticker}.parquet", columns=["date", "open", "high", "low", "close", "volume"],
                      filters=[("date", "<", datetime(2023, 1, 3))])
    d = t.to_pandas().sort_values("date").reset_index(drop=True)
    d["date"] = pd.to_datetime(d["date"]).dt.normalize()
    assert d.date.max() < SEAL_DATE
    return d

def daily_indicators(d):
    c, h, l, v, o = (d[k].astype(float) for k in ("close", "high", "low", "volume", "open"))
    sma20, sma50 = c.rolling(20).mean(), c.rolling(50).mean()
    macd = c.ewm(span=12, adjust=False).mean() - c.ewm(span=26, adjust=False).mean()
    hist = macd - macd.ewm(span=9, adjust=False).mean()
    dl = c.diff()
    ag = dl.clip(lower=0).ewm(alpha=1 / 14, adjust=False).mean()
    al = (-dl).clip(lower=0).ewm(alpha=1 / 14, adjust=False).mean()
    rsi = (100 - 100 / (1 + ag / al.replace(0, np.nan))).where(al > 0, np.where(ag > 0, 100.0, 50.0))
    sd = c.rolling(20).std(ddof=0)
    tr = pd.concat([h - l, (h - c.shift()).abs(), (l - c.shift()).abs()], axis=1).max(axis=1)
    atr = tr.ewm(alpha=1 / 14, adjust=False).mean()
    out = pd.DataFrame({"date": d.date})
    out["d_atr_pct"] = atr / c
    out["d_rsi14"] = rsi
    out["d_macd_hist_atr"] = hist / atr.replace(0, np.nan)
    out["d_close_sma20"] = c / sma20 - 1
    out["d_bb_pctb"] = (c - (sma20 - 2 * sd)) / (4 * sd).replace(0, np.nan)
    out["d_ret21"] = c / c.shift(21) - 1
    out["d_rv20"] = np.log(c).diff().rolling(20).std(ddof=0) * np.sqrt(252)
    out["d_dist_high252"] = c / h.rolling(252, min_periods=126).max() - 1
    out["_close"], out["_sma20"], out["_sma50"] = c, sma20, sma50
    out["_bb_up"], out["_bb_lo"], out["_rsi"] = sma20 + 2 * sd, sma20 - 2 * sd, rsi
    return out

PARITY = {}
if BARS_DIR.exists():
    for t in ["AAPL", "JPM", "KO"]:
        ind = daily_indicators(read_daily(t)).set_index("date")
        rows = FR[FR.ticker == t].set_index("asof_date")
        both = rows.index.intersection(ind.index)
        for c in ["d_atr_pct", "d_rsi14", "d_macd_hist_atr", "d_close_sma20", "d_bb_pctb", "d_ret21", "d_rv20",
                  "d_dist_high252"]:
            a, b = ind.loc[both, c].to_numpy(float), rows.loc[both, c].to_numpy(float)
            okm = np.isfinite(a) & np.isfinite(b)
            PARITY[(t, c)] = float(np.max(np.abs(a[okm] - b[okm]))) if okm.any() else np.nan
    par = pd.Series(PARITY).unstack()
    display(par.style.format("{:.1e}"))
    worst = float(np.nanmax(par.to_numpy()))
    print(f"largest absolute difference over all checks: {worst:.2e}"
          + (" (machine precision: the formulas reproduce the stored features exactly)" if worst < 1e-9 else ""))
    # a chart with the indicators the risk engine reads
    d = daily_indicators(read_daily("AAPL")).set_index("date").loc["2021-01-01":"2021-12-31"]
    fig, (a1, a2) = plt.subplots(2, 1, figsize=(11, 5), sharex=True, gridspec_kw={"height_ratios": [3, 1]})
    a1.plot(d.index, d._close, color="k", lw=1, label="close")
    a1.plot(d.index, d._sma20, label="SMA20"); a1.plot(d.index, d._sma50, label="SMA50")
    a1.fill_between(d.index, d._bb_lo, d._bb_up, alpha=.12, label="Bollinger(20, 2)")
    a1.set_title("AAPL 2021: the indicators the features and the risk engine use"); a1.legend(ncol=4, fontsize=8)
    a2.plot(d.index, d._rsi, color="#9467bd"); a2.axhline(70, ls="--", c="r", lw=.8); a2.axhline(30, ls="--", c="g", lw=.8)
    a2.set_ylabel("RSI 14"); plt.tight_layout(); plt.show()
else:
    print("Daily bars not found; the parity check is skipped.")

## 6. Baselines: the class prior and logistic regression, with a global-minimum certificate

* **Class prior.** Each horizon's training class shares, used as the forecast for every row. Every model must beat it.
* **Multinomial logistic regression** on the 80 features (train-median imputation with missing flags,
  standardisation on training rows, clipping at 8 standard deviations), with an L2 penalty of 1e-3 on the mean log
  loss: the logistic component of the frozen v1.1 blend (E09).

**"Did training reach the global minimum?"** For this model the question has an exact answer. The L2-penalised
multinomial log loss is strictly convex, so it has one global minimum. The cell below solves the same objective
from five random starts and checks that (a) all starts reach the same objective value and the same parameters,
(b) the gradient at the solution is numerically zero, and (c) the Hessian there is positive definite (its smallest
eigenvalue is at least the penalty). Together these certify the global minimum. A neural network has no such
certificate; section 9 shows what can be said instead.

In [ ]:
class ImputeScale:
    # train-median imputation + missing flags, then standardisation on training rows only
    def fit(self, X):
        self.cols, self.med = list(X.columns), X.median()
        self.miss = [c for c in self.cols if X[c].isna().any()]
        self.sc = StandardScaler().fit(self._z(X)); return self
    def _z(self, X):
        Z = X[self.cols].fillna(self.med)
        for c in self.miss:
            Z[c + "__missing"] = X[c].isna().astype(float)
        return Z.to_numpy(float)
    def transform(self, X):
        return np.clip(self.sc.transform(self._z(X)), -8, 8)

def sample(mask, n):
    ix = np.flatnonzero(mask)
    return ix if (n is None or len(ix) <= n) else np.sort(rng.choice(ix, n, replace=False))

N_TRAIN = 25_000 if QUICK else None
TR = sample(masks["train"], N_TRAIN)
PRE = ImputeScale().fit(FR.iloc[TR][NUM])
XS = PRE.transform(FR[NUM])                                # standardised features for every row
PRED = {}                                                  # model -> probabilities on EVAL rows, shape (n, 6, 3)

prior = np.zeros((len(EVI), 6, 3))
logit = np.zeros((len(EVI), 6, 3))
LOGIT_MODELS = {}
t0 = time.time()
for j, h in enumerate(H6):
    ytr = Y[TR, j]; ok = ytr >= 0
    p = np.bincount(ytr[ok], minlength=3) / ok.sum()
    prior[:, j] = p
    m = LogisticRegression(C=1.0 / (1e-3 * ok.sum()), max_iter=3000)     # sklearn's C: penalty 1e-3 on the mean loss
    m.fit(XS[TR][ok], ytr[ok])
    logit[:, j] = m.predict_proba(XS[EVI])
    LOGIT_MODELS[h] = m
PRED["prior"], PRED["logistic"] = prior, logit
print(f"prior and logistic fitted for 6 horizons in {time.time() - t0:.0f}s")

# ---- the certificate (5D horizon; a 20,000-row training sample keeps the exact Hessian cheap) ----------------------
def certificate(X, y, lam=1e-3, starts=5):
    # objective: mean multinomial log loss + lam/2 * ||W||^2 (all parameters penalised, as in E09)
    n, d = X.shape
    Xb = np.hstack([X, np.ones((n, 1))])
    Yo, k = np.eye(3)[y], (d + 1) * 3
    def probs(w):
        Zl = Xb @ w.reshape(d + 1, 3); Zl -= Zl.max(1, keepdims=True)
        P = np.exp(Zl); return P / P.sum(1, keepdims=True)
    def f_g(w):
        P, W = probs(w), w.reshape(d + 1, 3)
        f = -np.mean(np.log(P[np.arange(n), y] + 1e-300)) + 0.5 * lam * np.sum(W ** 2)
        return f, (Xb.T @ (P - Yo) / n + lam * W).ravel()
    def hessian(w):                                             # exact: data term (positive semi-definite) + lam * I
        P, Hm = probs(w), np.zeros((k, k))
        for a in range(3):
            for b in range(3):
                Hm[a::3, b::3] = (Xb * (P[:, a] * ((a == b) - P[:, b]))[:, None]).T @ Xb / n
        return Hm + lam * np.eye(k)
    sols = []
    for s in range(starts):                                     # five random starts, far apart
        w = optimize.minimize(f_g, np.random.default_rng(100 + s).normal(0, 1, k), jac=True, method="L-BFGS-B",
                              options={"maxiter": 20000, "ftol": 1e-15, "gtol": 1e-10}).x
        for _ in range(3):                                      # Newton steps with the exact Hessian (quadratic convergence)
            w = w - np.linalg.solve(hessian(w), f_g(w)[1])
        sols.append(w)
    fs = np.array([f_g(w)[0] for w in sols]); best = sols[int(fs.argmin())]
    eig = np.linalg.eigvalsh(hessian(best))
    gn = float(np.linalg.norm(f_g(best)[1]))
    dist = float(max(np.abs(w - best).max() for w in sols))
    ok = bool(np.ptp(fs) < 1e-10 and gn < 1e-6 and dist < 1e-5 and eig.min() > 0)
    return {"objective at each start": ", ".join(f"{f:.12f}" for f in fs), "objective range": f"{np.ptp(fs):.1e}",
            "max parameter distance between starts": f"{dist:.1e}", "gradient norm at the solution": f"{gn:.1e}",
            "Hessian smallest eigenvalue": f"{eig.min():.6f} (> 0: strictly convex)",
            "Hessian largest eigenvalue": f"{eig.max():.3f}", "certified global minimum": ok}

j5 = H6.index("5D")
cs = sample(masks["train"] & (Y[:, j5] >= 0), 4_000 if QUICK else 20_000)
CERT = certificate(XS[cs], Y[cs, j5])
display(pd.Series(CERT, name="5D logistic objective, 5 random starts").to_frame())
assert CERT["certified global minimum"], "the certificate failed"

## 7. LightGBM: gradient-boosted trees, with training and validation loss curves

One multiclass booster per horizon on the same 80 features (trees need no scaling). Settings are the project's fixed
defaults (learning rate 0.05, 31 leaves, at least 400 rows per leaf, L2 = 5, bagging 0.8). Training stops when the
loss on the **inner** block (2016) has not improved for 100 rounds: the validation years are never used to stop or
tune. The curves show the training loss falling steadily while the held-out loss flattens early; the gap between
them is the variance the early stop controls.

In [ ]:
LGB = {"objective": "multiclass", "num_class": 3, "learning_rate": 0.05, "num_leaves": 31, "min_data_in_leaf": 400,
       "feature_fraction": 0.8, "bagging_fraction": 0.8, "bagging_freq": 1, "lambda_l2": 5.0, "verbose": -1,
       "deterministic": True, "force_row_wise": True, "seed": SEED, "num_threads": os.cpu_count() or 4}
INN = sample(masks["inner"], 6_000 if QUICK else None)
VAL = np.flatnonzero(masks["valid"])
gb = np.zeros((len(EVI), 6, 3))
CURVES, IMP, BEST_IT = {}, {}, {}
Xdf = FR[NUM].astype(np.float32)

def stop_on_inner(patience=100, name="inner (early stop)"):
    # early stopping that watches ONLY the inner block; the validation curve is recorded for display, never used
    best = {"v": float("inf"), "it": 0}
    def _cb(env):
        for ds, metric, val, _ in env.evaluation_result_list:
            if ds != name:
                continue
            if val < best["v"] - 1e-12:
                best.update(v=val, it=env.iteration)
            elif env.iteration - best["it"] >= patience:
                raise lgb.callback.EarlyStopException(best["it"], [(name, metric, best["v"], False)])
    _cb.order = 30
    return _cb

t0 = time.time()
for j, h in enumerate(H6):
    ok_tr, ok_in, ok_va = Y[TR, j] >= 0, Y[INN, j] >= 0, Y[VAL, j] >= 0
    dtr = lgb.Dataset(Xdf.iloc[TR[ok_tr]], Y[TR[ok_tr], j], free_raw_data=False)
    din = lgb.Dataset(Xdf.iloc[INN[ok_in]], Y[INN[ok_in], j], reference=dtr)
    dva = lgb.Dataset(Xdf.iloc[VAL[ok_va]], Y[VAL[ok_va], j], reference=dtr)
    ev = {}
    bst = lgb.train(LGB, dtr, num_boost_round=300 if QUICK else 1500, valid_sets=[dtr, din, dva],
                    valid_names=["train", "inner (early stop)", "valid (shown only)"],
                    callbacks=[stop_on_inner(100), lgb.record_evaluation(ev)])
    CURVES[h], BEST_IT[h] = ev, bst.best_iteration
    gb[:, j] = bst.predict(Xdf.iloc[EVI], num_iteration=bst.best_iteration)
    IMP[h] = pd.Series(bst.feature_importance("gain"), index=NUM)
PRED["lgbm"] = gb
print(f"LightGBM, 6 horizons, {time.time() - t0:.0f}s; best iterations:", BEST_IT)

fig, axs = plt.subplots(2, 3, figsize=(13, 6.2))
for ax, h in zip(axs.ravel(), H6):
    for name, c in (("train", "#1f77b4"), ("inner (early stop)", "#ff7f0e"), ("valid (shown only)", "#2ca02c")):
        ax.plot(CURVES[h][name]["multi_logloss"], color=c, label=name, lw=1.4)
    ax.axvline(BEST_IT[h], color="k", ls=":", lw=1)
    ax.set_title(f"{h}: multiclass log loss by boosting round"); ax.set_xlabel("round")
axs[0, 0].legend(fontsize=8); plt.tight_layout(); plt.show()

imp = pd.DataFrame(IMP)
imp = imp.div(imp.sum(0), axis=1).mean(1).sort_values(ascending=False).head(20)
fig, ax = plt.subplots(figsize=(8, 5)); imp[::-1].plot.barh(ax=ax, color="#2ca02c")
ax.set_title("LightGBM: share of total gain, mean over horizons (top 20)"); plt.tight_layout(); plt.show()

## 8. The multimodal fusion network (GPU)

Six input blocks, each with a presence flag (a missing block is zeroed **and** flagged, so absence is never read as
neutral evidence):

| Block | Content | Dimension |
|---|---|---|
| `img_W`, `img_D`, `img_H` | weekly, daily and hourly chart images through a frozen ImageNet ConvNeXt-Tiny, reduced to 64 principal components per view (fitted on training rows) | 3 x 64 |
| `num` | the 80 indicator features | 80 |
| `news` | aggregated company-news representation (TF-IDF/SVD, Loughran-McDonald tone, counts) available before the decision time | 75 |
| `sec` | SEC filing features (8-K, 10-Q/10-K timing, items, text) inside the SEC study's coverage | 91 |

**Architecture (gated fusion, the variant selected in E03).** Each block is projected to 64 dimensions
(Linear-GELU-Dropout-Linear-LayerNorm) and tagged with a learned type embedding. A gating network scores the present
blocks; a softmax over them weights the projected vectors into one representation. Two heads read it: a classifier
for 6 horizons x 3 classes and quantile heads (10%, 50%, 90%) of the volatility-scaled return.

**Loss.** Cross-entropy over the six horizons (missing labels ignored) plus 0.2 x the pinball loss of the quantiles.
**Training.** AdamW (learning rate 3e-4 with a cosine schedule, weight decay 1e-4, dropout 0.3: the E06 optimum),
modality dropout 0.15, batches of 1,024, early stopping on the inner block (patience 4), three seeds averaged.

In [ ]:
img = np.load(NB / "img_pca64_convnext_tiny.npy").astype(np.float32)        # (236520, 192), rows as FR
feats = {k: pd.read_parquet(NB / "features" / f"{k}_features.parquet") for k in ("news", "sec")}
news = FR[["sample_id"]].merge(feats["news"], left_on="sample_id", right_index=True, how="left")
sec = FR[["sample_id"]].merge(feats["sec"], left_on="sample_id", right_index=True, how="left")
NEWS_COLS = [c for c in news.columns if c.startswith("news_") and c not in ("news_present", "news_n_selected_no_text")]
SEC_COLS = [c for c in sec.columns if c.startswith("sec_") and c != "sec_covered"]

def standardise(X, fit_rows):
    mu, sd = np.nanmean(X[fit_rows], 0), np.nanstd(X[fit_rows], 0)
    sd = np.where(np.isfinite(sd) & (sd > 1e-8), sd, 1.0)
    return np.clip(np.nan_to_num((X - np.nan_to_num(mu)) / sd, nan=0.0), -8, 8).astype(np.float32)

blocks, present = {}, {}
for k, v in enumerate("WDH"):
    need = {"W": 42, "D": 51, "H": 56}[v]                      # at least 40% of the window's bars must exist
    present[f"img_{v}"] = (FR[f"n_valid_{v}"].to_numpy() >= need).astype(np.float32)
    blocks[f"img_{v}"] = standardise(img[:, 64 * k: 64 * (k + 1)], TR) * present[f"img_{v}"][:, None]
blocks["num"] = standardise(FR[NUM].to_numpy(np.float64), TR); present["num"] = np.ones(len(FR), np.float32)
for name, df, cols, flag in (("news", news, NEWS_COLS, "news_present"), ("sec", sec, SEC_COLS, "sec_covered")):
    pr = df[flag].fillna(0).to_numpy(np.float32)
    fit = TR[pr[TR] > 0]
    blocks[name] = standardise(df[cols].to_numpy(np.float64), fit) * pr[:, None]
    present[name] = pr
NAMES = list(blocks)
MASK = np.stack([present[n] for n in NAMES], 1)
print({n: blocks[n].shape[1] for n in NAMES})
print("share of rows with the block present:", {n: round(float(present[n].mean()), 3) for n in NAMES})

class Projector(nn.Module):
    def __init__(self, d_in, d, p):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_in, d), nn.GELU(), nn.Dropout(p), nn.Linear(d, d), nn.LayerNorm(d))
    def forward(self, x):
        return self.net(x)

class GatedFusion(nn.Module):
    def __init__(self, dims, n_h=6, d=64, p=0.3):
        super().__init__()
        self.names = list(dims)
        self.proj = nn.ModuleDict({m: Projector(k, d, p) for m, k in dims.items()})
        self.type_emb = nn.Parameter(torch.randn(len(dims), d) * 0.02)
        self.gate = nn.Sequential(nn.Linear(d, d), nn.Tanh(), nn.Linear(d, 1))
        self.cls, self.q, self.n_h = nn.Linear(d, n_h * 3), nn.Linear(d, n_h * 3), n_h
    def forward(self, xs, mask):
        Hm = torch.stack([self.proj[m](xs[m]) for m in self.names], 1) + self.type_emb[None]
        Hm = Hm * mask[..., None]
        g = self.gate(Hm).squeeze(-1).masked_fill(mask == 0, -1e9).softmax(-1)      # weights over present blocks
        z = (g[..., None] * Hm).sum(1)
        return self.cls(z).view(-1, self.n_h, 3), self.q(z).view(-1, self.n_h, 3), g

QS = torch.tensor([0.1, 0.5, 0.9])
def fusion_loss(logits, quant, y, z, qw=0.2):
    ce = F.cross_entropy(logits.reshape(-1, 3), y.reshape(-1), ignore_index=-1, reduction="sum") / (y >= 0).sum().clamp(min=1)
    zm = torch.isfinite(z); zz = torch.nan_to_num(z)
    d = zz[..., None] - quant
    qs = QS.to(z.device)
    pin = (torch.maximum(qs * d, (qs - 1) * d) * zm[..., None]).sum() / (zm.sum().clamp(min=1) * 3)
    return ce + qw * pin, ce

def to_dev(ix, B=None):
    # B = (blocks, names, mask) of another block set (section 8C); default: this section's six blocks
    bl, nm, mk = B or (blocks, NAMES, MASK)
    return ({n: torch.from_numpy(bl[n][ix]).to(DEVICE) for n in nm}, torch.from_numpy(mk[ix]).to(DEVICE),
            torch.from_numpy(Y[ix]).to(DEVICE), torch.from_numpy(ZV[ix]).to(DEVICE))

@torch.no_grad()
def eval_ce(model, data, bs=8192):
    model.eval(); xs, m, y, z = data; tot, n = 0.0, 0
    for i in range(0, len(m), bs):
        lg, qu, _ = model({k: v[i:i + bs] for k, v in xs.items()}, m[i:i + bs])
        yy = y[i:i + bs]
        tot += F.cross_entropy(lg.reshape(-1, 3), yy.reshape(-1), ignore_index=-1, reduction="sum").item()
        n += int((yy >= 0).sum())
    return tot / max(n, 1)

@torch.no_grad()
def predict_probs(model, ix, bs=8192, B=None):
    model.eval(); xs, m, _, _ = to_dev(ix, B); out = []
    for i in range(0, len(m), bs):
        lg, _, _ = model({k: v[i:i + bs] for k, v in xs.items()}, m[i:i + bs])
        out.append(lg.softmax(-1).float().cpu().numpy())
    return np.concatenate(out)

def train_fusion(seed, epochs=30, bs=1024, lr=3e-4, wd=1e-4, p_drop=0.3, mod_drop=0.15, patience=4, B=None):
    torch.manual_seed(seed); g = torch.Generator(device="cpu").manual_seed(seed)
    bl, nm, _ = B or (blocks, NAMES, MASK)
    model = GatedFusion({n: bl[n].shape[1] for n in nm}, p=p_drop).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    tr, inn, va = to_dev(TR, B), to_dev(INN, B), to_dev(VAL, B)
    hist, best, best_state, bad = [], 1e9, None, 0
    for ep in range(epochs):
        model.train(); perm = torch.randperm(len(TR), generator=g).to(DEVICE); run, nb = 0.0, 0
        for i in range(0, len(TR), bs):
            b = perm[i:i + bs]
            m = tr[1][b]
            drop = (torch.rand(m.shape, device=DEVICE) < mod_drop).float()
            m2 = m * (1 - drop); empty = m2.sum(1) == 0; m2[empty] = m[empty]      # never hide every block
            lg, qu, _ = model({k: v[b] for k, v in tr[0].items()}, m2)
            loss, ce = fusion_loss(lg, qu, tr[2][b], tr[3][b])
            opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
            run += ce.item(); nb += 1
        sched.step()
        e_in, e_va = eval_ce(model, inn), eval_ce(model, va)
        hist.append({"epoch": ep + 1, "train CE": run / nb, "inner CE (early stop)": e_in, "valid CE (shown only)": e_va,
                     "learning rate": sched.get_last_lr()[0]})
        if e_in < best - 1e-5:
            best, bad = e_in, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= patience:
                break
    model.load_state_dict(best_state)
    return model, pd.DataFrame(hist)

SEEDS = [7, 11] if QUICK else [7, 11, 23]
FUSION, HIST = {}, {}
t0 = time.time()
for s in SEEDS:
    FUSION[s], HIST[s] = train_fusion(s, epochs=2 if QUICK else 30)
    print(f"seed {s}: {len(HIST[s])} epochs, best inner CE {HIST[s]['inner CE (early stop)'].min():.5f} "
          f"({time.time() - t0:.0f}s)")
PRED["fusion"] = np.mean([predict_probs(FUSION[s], EVI) for s in SEEDS], 0)

fig, axs = plt.subplots(1, len(SEEDS), figsize=(4.6 * len(SEEDS), 3.4), squeeze=False)
for ax, s in zip(axs[0], SEEDS):
    h = HIST[s]
    for c, col in (("train CE", "#1f77b4"), ("inner CE (early stop)", "#ff7f0e"), ("valid CE (shown only)", "#2ca02c")):
        ax.plot(h.epoch, h[c], marker="o", ms=3, label=c, color=col)
    ax.axvline(h.loc[h["inner CE (early stop)"].idxmin(), "epoch"], color="k", ls=":", lw=1)
    ax.set_title(f"fusion seed {s}: mean 6-horizon cross-entropy"); ax.set_xlabel("epoch")
axs[0, 0].legend(fontsize=8); plt.tight_layout(); plt.show()

with torch.no_grad():                                     # what the gates attend to (a diagnostic, not a cause)
    m0 = FUSION[SEEDS[0]]; xs, mk, _, _ = to_dev(VAL[:20000])
    _, _, gates = m0(xs, mk)
gw = pd.Series(gates.float().mean(0).cpu().numpy(), index=NAMES)
fig, ax = plt.subplots(figsize=(6, 2.6)); gw.plot.bar(ax=ax, color="#d62728")
ax.set_title("mean gate weight per block, valid rows (model diagnostic)"); plt.tight_layout(); plt.show()

## 8A. The language branch: a chronologically consistent transformer reads the SEC filings (GPU)

Section 8 reads text through classical features only (TF-IDF components and Loughran-McDonald tone). Here a transformer
reads the filings themselves.

**Why ChronoBERT, not SEC-BERT or FinBERT.** A language model that has read text from the years it is evaluated on
carries look-ahead: it may "know" how a company's story ended. ChronoBERT (He, Lv, Manela and Wu, 2025) is a
ModernBERT-base model pretrained only on text dated before a cutoff. NEXUS uses the 2010-12-31 cutoff, so the encoder has
never read anything from the early-stopping (2016), validation (2017-18), calibration (2019) or test (2020-22) years.
SEC-BERT was pretrained on 10-K filings up to 2019, which would include the validation years' own filings.

**What is encoded.** Every filing that is some sample's latest 8-K (30 days) or latest 10-Q/10-K (120 days) at its
decision time: 68,853 filings of the SEC study's 500 firms. The text is the SEC study's event text (the filing's main
sections, with firm names masked as `[FIRM]`). Safe-harbour paragraphs are dropped, and the first 256 tokens are
encoded. The vector is the masked mean of the last layer, 768 numbers.

**Reproducibility.** The documented run (E28) encoded these filings on the project PC's CPU. This cell re-encodes them
on the GPU and checks that the vectors match. Every filing is re-encoded on a GPU (a few minutes on an A100 or L4);
a random 400 on a CPU, and 1,000 (GPU) or 300 (CPU) in QUICK mode. The check is the cosine after removing the
pre-2016 mean vector (raw mean-pooled vectors of unrelated filings already have cosine near 0.99). bfloat16 on a GPU
does not give bit-identical vectors; on 48 filings, bfloat16 against the stored float32 vectors gave a median centred
cosine of 0.9998 (minimum 0.973). The cell requires a median above 0.99.

In [ ]:
NLP = NB / "nlp"
EVM = pd.read_parquet(NLP / "sec_tx_events.parquet")                     # one row per encoded filing
E_REF = np.load(NLP / "sec_tx_event_emb.npy")                             # the vectors E28 used (CPU, float16)
TXT = pd.read_parquet(NLP / "filing_texts.parquet").set_index("event_id").text.reindex(EVM.event_id).fillna("")
EVM["available_utc"] = pd.to_datetime(EVM.available_utc, utc=True)
assert EVM.available_utc.max() < pd.Timestamp(SEAL_DATE, tz="UTC"), "a sealed filing is in the inputs"
print(f"{len(EVM):,} filings, {EVM.event_type.value_counts().to_dict()}")
display(pd.DataFrame({"filing": EVM.event_type.iloc[[0, 1, 2]].values,
                      "available (UTC)": EVM.available_utc.iloc[[0, 1, 2]].astype(str).values,
                      "cleaned text, first 160 characters": [t[:160] for t in TXT.iloc[[0, 1, 2]]]}))

CHRONO, CHRONO_REV = "manelalab/chrono-bert-v1-20101231", "1520cf239e7922584ce7762af9e5040e8edd9374"
try:
    from transformers import AutoModel, AutoTokenizer
    import transformers
    assert tuple(int(x) for x in transformers.__version__.split(".")[:2]) >= (4, 48)      # ModernBERT support
except (ImportError, AssertionError):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers>=4.48"], check=True)
    from transformers import AutoModel, AutoTokenizer
TOK = AutoTokenizer.from_pretrained(CHRONO, revision=CHRONO_REV)
ENC = AutoModel.from_pretrained(CHRONO, revision=CHRONO_REV).eval().to(DEVICE)
if DEVICE == "cuda":
    ENC = ENC.to(torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16)

@torch.inference_mode()
def chrono_encode(texts, bs=None, max_tokens=256):
    bs = bs or (256 if DEVICE == "cuda" else 16)
    order = np.argsort([len(t) for t in texts], kind="stable")
    out = np.zeros((len(texts), ENC.config.hidden_size), np.float32)
    for i in range(0, len(texts), bs):
        ix = order[i:i + bs]
        b = TOK([texts[k] for k in ix], padding=True, truncation=True, max_length=max_tokens, return_tensors="pt").to(DEVICE)
        hs = ENC(**b).last_hidden_state.float()
        m = b["attention_mask"].unsqueeze(-1).float()
        out[ix] = ((hs * m).sum(1) / m.sum(1).clamp(min=1)).cpu().numpy()
    return out

has_text = np.flatnonzero(EVM.n_chars_clean.to_numpy() > 0)
n_enc = (1_000 if DEVICE == "cuda" else 300) if QUICK else (len(has_text) if DEVICE == "cuda" else 400)
SEL = np.sort(rng.choice(has_text, n_enc, replace=False)) if n_enc < len(has_text) else has_text
t0 = time.time()
E_NEW = chrono_encode(TXT.iloc[SEL].tolist())
dt = time.time() - t0
MU_TX = np.load(NLP / "sec_tx_pca.npz", allow_pickle=True)["mean"].astype(np.float32)
# cosine after removing the pre-2016 mean: raw mean-pooled BERT vectors all point the same way (cosine ~0.99 between
# unrelated filings), so only the centred cosine is a real parity test
a, b = E_NEW - MU_TX, E_REF[SEL].astype(np.float32) - MU_TX
COS = (a * b).sum(1) / (np.linalg.norm(a, axis=1) * np.linalg.norm(b, axis=1))
print(f"encoded {len(SEL):,} filings on {DEVICE} in {dt:.0f} s ({len(SEL) / dt:.0f} per second); centred cosine with "
      f"the stored vectors: median {np.median(COS):.5f}, 1st percentile {np.quantile(COS, .01):.5f}")
assert np.median(COS) > 0.99, "the re-encoding does not reproduce the stored vectors"

In [ ]:
# what the vectors capture: two principal components (fitted on pre-2016 filings) and point-in-time nearest neighbours
PCA_TX = np.load(NLP / "sec_tx_pca.npz", allow_pickle=True)
OKV = np.isfinite(E_REF).all(1)
Z2 = np.full((len(E_REF), 2), np.nan)
Z2[OKV] = (E_REF[OKV].astype(np.float64) - PCA_TX["mean"]) @ PCA_TX["components"][:2].T
ITEM = {"2.02": "8-K results", "5.02": "8-K officers", "1.01": "8-K agreement", "8.01": "8-K other events",
        "7.01": "8-K Reg FD", "2.01": "8-K acquisition"}
KIND = np.where(EVM.event_type == "8K", EVM.item_main.map(ITEM).fillna("8-K other items"),
                EVM.event_type.str.replace("periodic_", "").str.replace("10", "10-"))
fig, ax = plt.subplots(figsize=(8, 5.2))
sub = rng.choice(np.flatnonzero(OKV), min(20_000, int(OKV.sum())), replace=False)
for k, c in zip(pd.Series(KIND[sub]).value_counts().index[:8], plt.cm.tab10.colors):
    m = sub[KIND[sub] == k]
    ax.scatter(Z2[m, 0], Z2[m, 1], s=3, alpha=.35, color=c, label=k)
ax.set_title("ChronoBERT filing vectors: the first two principal components")
ax.set_xlabel("component 1"); ax.set_ylabel("component 2")
ax.legend(markerscale=5, fontsize=8, ncol=4, loc="upper center", bbox_to_anchor=(0.5, -0.14), frameon=False)
plt.tight_layout(); plt.show()

q = int(np.flatnonzero(OKV & (EVM.item_main == "2.02").to_numpy() & (EVM.available_utc.dt.year == 2018).to_numpy())[0])
past = np.flatnonzero(OKV & (EVM.available_utc < EVM.available_utc.iloc[q]).to_numpy())      # only earlier filings
V = E_REF[past].astype(np.float32) - MU_TX; qv = E_REF[q].astype(np.float32) - MU_TX
sim = V @ qv / (np.linalg.norm(V, axis=1) * np.linalg.norm(qv))
nn_ix = past[np.argsort(-sim)[:3]]
display(Markdown(f"**Nearest earlier filings to an 8-K results filing of {EVM.available_utc.iloc[q].date()}** "
                 "(cosine of the centred ChronoBERT vectors; firm names are masked in the SEC study's text)"))
display(pd.DataFrame({"filing": [KIND[q]] + list(KIND[nn_ix]),
                      "date": [str(EVM.available_utc.iloc[i].date()) for i in [q] + list(nn_ix)],
                      "cosine": [1.0] + [float(s) for s in np.sort(sim)[::-1][:3]],
                      "text, first 140 characters": [TXT.iloc[i][:140] for i in [q] + list(nn_ix)]}))

## 8B. The SEC price-impact expert: what the SEC study learned, as evidence for NEXUS

The SEC filings study labelled each filing by what happened to the stock after it became public. `T_trade` enters at
the first tradable price after the decision time; `label3` is UP, SIDEWAYS or DOWN of the abnormal return at 1 and 5
sessions. Its headline finding was that filings predict **whether** a stock moves (AUC 0.635 on its locked test), not
**which way** (0.494).

Here that knowledge becomes a model NEXUS can use:
* **Model.** LightGBM on the study's own event features. Two families are left out: `meta_year` (review finding N2),
  and the accounting features (finding N1: their availability rule is wrong before 2009).
* **Fitting.** Fitted on events before 2016; trees are chosen on 2016 events.
* **Predictions.** Every event gets one. Events before 2016 are predicted out of fold, so no NEXUS training row
  sees a prediction fitted on its own event.
* **Sealed period.** No label with an exit after 2018 is read.

The cell refits the expert and checks its AUCs and predictions against the documented build.

In [ ]:
TE_ = pd.read_parquet(NLP / "sec_expert_table.parquet")
SI_META = json.loads((NLP / "sec_expert_meta.json").read_text(encoding="utf-8"))
SI_FEAT = SI_META["features"]
SI_PARAMS = {**SI_META["params"], "num_threads": os.cpu_count() or 2}       # the documented fixed parameters
at = pd.to_datetime(TE_.available_utc, utc=True)
F16, F17 = pd.Timestamp("2016-01-01", tz="UTC"), pd.Timestamp("2017-01-01", tz="UTC")
XE = TE_[SI_FEAT].to_numpy(np.float32)
SI_MODELS, SI_AUC, SI_NEW = {}, [], {}
for h in (1, 5):
    y = TE_[f"y_h{h}"].to_numpy(); ex = pd.to_datetime(TE_[f"exit_h{h}"], utc=True)
    tr_ = ((at < F16) & (ex < F16)).to_numpy() & np.isfinite(y)
    es_ = ((at >= F16) & (at < F17) & (ex < F17)).to_numpy() & np.isfinite(y)
    d_tr = lgb.Dataset(XE[tr_], y[tr_].astype(int)); d_es = lgb.Dataset(XE[es_], y[es_].astype(int), reference=d_tr)
    m = lgb.train(SI_PARAMS, d_tr, num_boost_round=2000, valid_sets=[d_es],
                  callbacks=[lgb.early_stopping(100, verbose=False)])
    SI_MODELS[h] = m
    va_ = at.dt.year.isin([2017, 2018]).to_numpy() & np.isfinite(y)
    p = m.predict(XE[va_], num_iteration=m.best_iteration)
    SI_NEW[h] = (np.flatnonzero(va_), p)
    mv = (y[va_] != 1).astype(int); ud = y[va_] != 1
    SI_AUC.append({"horizon": f"{h} session(s)", "trees": m.best_iteration, "training events": int(tr_.sum()),
                   "valid events (2017-18)": int(va_.sum()), "AUC: does it move?": roc_auc_score(mv, 1 - p[:, 1]),
                   "AUC: up vs down": roc_auc_score((y[va_][ud] == 2).astype(int), p[ud, 2] - p[ud, 0]),
                   "documented AUC (move)": SI_META["diagnostics"][f"h{h}_valid_2017_2018"]["auc_move"]})
display(pd.DataFrame(SI_AUC).style.format(precision=4))

DOC = pd.read_parquet(NLP / "sec_impact_event_predictions.parquet")
ix1, p1 = SI_NEW[1]
diff = np.abs(p1[:, 1] - DOC.pside_h1.reindex(TE_.index[ix1]).to_numpy())
print(f"h1 P(sideways) on 2017-18 events, refit here vs the documented build: median |diff| {np.median(diff):.1e}, "
      f"max {diff.max():.1e} (LightGBM is deterministic within one version and machine type)")

imp = pd.Series(SI_MODELS[1].feature_importance("gain"), index=SI_FEAT).sort_values().tail(15)
fig, axs = plt.subplots(1, 2, figsize=(13, 4.2))
axs[0].barh(imp.index, imp / imp.sum(), color="#8c564b"); axs[0].set_title("expert, 1 session: top features (gain share)")
yv = TE_["y_h1"].to_numpy()[ix1]
dec = pd.qcut(1 - p1[:, 1], 10, labels=False, duplicates="drop")
rate = pd.Series(yv != 1).groupby(dec).mean()
axs[1].plot(rate.index + 1, rate.to_numpy(), marker="o", color="#8c564b")
axs[1].set_xlabel("decile of predicted P(move), 2017-18 events"); axs[1].set_ylabel("share that moved")
axs[1].set_title("does the expert rank filings by how much they move the price?")
plt.tight_layout(); plt.show()

## 8C. Vision + NLP: one fusion network over charts, indicators and language (E28)

E28 was pre-registered before any of its features existed (`docs/V3_RESEARCH.md`, decision D-053). It adds three
language blocks to section 8's six:

| Block | What it is | Present for |
|---|---|---|
| `news_tx` | FinBERT tone and a 16-component embedding of the company's headlines before the decision time | samples with news |
| `sec_tx` | ChronoBERT vectors (section 8A) of the latest 8-K and 10-Q/10-K, 32 components each | samples with a recent filing |
| `sec_impact` | the SEC expert's P(move) and P(up) - P(down) for the latest filing (section 8B) | a filing within 14 days |

**The test.** MM (all nine blocks) is compared with BASE (section 8's network: the same settings and seeds). It is also
compared with PLACEBO: MM with the three new blocks moved to another stock of the same date. The placebo keeps their
distribution but breaks their link to the stock. The language blocks are **kept** only if both validation intervals
lie above zero; the test years are reported once, after that decision.

This cell retrains MM and PLACEBO here; BASE is section 8's model. The documented run (`experiments/20261113T035556_e28_vision_nlp_fusion_b1f899`) used the
same design with the package code (`scripts/experiments/e28_vision_nlp_fusion.py`). Its table follows, with the
descriptive variants.

In [ ]:
fb = pd.read_parquet(NB / "features" / "finbert_features.parquet").set_index("sample_id")
stx = pd.read_parquet(NB / "features" / "sec_tx_features.parquet")
sim_ = pd.read_parquet(NB / "features" / "sec_impact_features.parquet")
J = FR[["sample_id"]].join(fb, on="sample_id").join(stx, on="sample_id").join(sim_, on="sample_id")
NEWB = {"news_tx": ([c for c in fb.columns if c != "fb_n"], J.fb_n.fillna(0).to_numpy() > 0),
        "sec_tx": ([c for c in stx.columns if c != "sec_tx_present"], J.sec_tx_present.fillna(0).to_numpy() > 0),
        "sec_impact": ([c for c in sim_.columns if c != "si_present"], J.si_present.fillna(0).to_numpy() > 0)}
blocks2, present2 = dict(blocks), dict(present)
for n, (cols, pr) in NEWB.items():
    pr = pr.astype(np.float32)
    blocks2[n] = standardise(J[cols].to_numpy(np.float64), TR[pr[TR] > 0]) * pr[:, None]
    present2[n] = pr
print({n: (blocks2[n].shape[1], round(float(present2[n][masks["valid"]].mean()), 3)) for n in NEWB},
      "(dimension, share present on valid)")

PERM = np.arange(len(FR)); r_pl = np.random.default_rng(11)                 # placebo: within-date shuffle across stocks
for idx in FR.groupby("asof_date").indices.values():
    PERM[idx] = idx[r_pl.permutation(len(idx))]

def block_set(names, placebo=False):
    bl = {n: (blocks2[n][PERM] if placebo and n in NEWB else blocks2[n]) for n in names}
    mk = np.stack([(present2[n][PERM] if placebo and n in NEWB else present2[n]) for n in names], 1)
    return bl, list(names), mk

MM_NAMES = list(NAMES) + list(NEWB)
SETS = {"MM": block_set(MM_NAMES), "PLACEBO": block_set(MM_NAMES, placebo=True)}
FUS2, HIST2, PV = {}, {}, {}
t0 = time.time()
for k, B in SETS.items():
    for s in SEEDS:
        FUS2[(k, s)], HIST2[(k, s)] = train_fusion(s, epochs=2 if QUICK else 30, B=B)
    PV[k] = np.mean([predict_probs(FUS2[(k, s)], EVI, B=B) for s in SEEDS], 0)
    print(f"{k}: {len(SEEDS)} seeds trained ({time.time() - t0:.0f} s)")
PV["BASE"] = PRED["fusion"]
PRED["fusion_mm"] = PV["MM"]

POSV = {i: k for k, i in enumerate(EVI)}
def rows_of(mask):
    return np.array([POSV[i] for i in np.flatnonzero(mask) if i in POSV])
YE_ = Y[EVI]
def mean_ll6(P, ix):
    return float(np.mean([log_loss(YE_[ix[YE_[ix, j] >= 0], j], np.clip(P[ix[YE_[ix, j] >= 0], j], 1e-7, 1),
                                   labels=[0, 1, 2]) for j in range(6)]))

def paired_gain(a, b, ix, n_boot=300 if QUICK else 1000, seed=7):
    # mean LL(b) - mean LL(a) over the six horizons; month-block bootstrap (positive = a better)
    months = pd.to_datetime(FR.asof_date.to_numpy()[EVI[ix]]).to_period("M").astype(str)
    groups = pd.Series(np.arange(len(ix))).groupby(months).indices
    keys = list(groups); r = np.random.default_rng(seed)
    est = mean_ll6(PV[b], ix) - mean_ll6(PV[a], ix)
    bs = []
    for _ in range(n_boot):
        pick = np.concatenate([groups[keys[k]] for k in r.integers(0, len(keys), len(keys))])
        bs.append(mean_ll6(PV[b], ix[pick]) - mean_ll6(PV[a], ix[pick]))
    return {"gain": est, "low": float(np.quantile(bs, .025)), "high": float(np.quantile(bs, .975))}

IXV, IXT = rows_of(masks["valid"]), rows_of(masks["test"])
G28 = pd.DataFrame({("valid 2017-18", "MM vs BASE"): paired_gain("MM", "BASE", IXV),
                    ("valid 2017-18", "MM vs PLACEBO"): paired_gain("MM", "PLACEBO", IXV),
                    ("test 2020-22 (after the decision)", "MM vs BASE"): paired_gain("MM", "BASE", IXT),
                    ("test 2020-22 (after the decision)", "MM vs PLACEBO"): paired_gain("MM", "PLACEBO", IXT)}).T
KEPT_HERE = bool((G28.loc["valid 2017-18", "low"] > 0).all())
display(Markdown(f"**Mean six-horizon log-loss gain of the vision + NLP network (positive = better; 95% month-block "
                 f"intervals).** Keep rule on validation: **{'KEPT' if KEPT_HERE else 'NOT KEPT'}** in this run."))
display(G28.style.format("{:+.5f}"))

In [ ]:
# where the network looks, the loss curves and the confusion matrices of the vision + NLP model
B_MM = SETS["MM"]
with torch.no_grad():
    xs, mk, _, _ = to_dev(VAL[:20000], B_MM)
    _, _, g = FUS2[("MM", SEEDS[0])](xs, mk)
GW = pd.Series(g.float().mean(0).cpu().numpy(), index=MM_NAMES)
GROUP = {"img_W": "vision", "img_D": "vision", "img_H": "vision", "num": "indicators", "news": "language",
         "sec": "language", "news_tx": "language", "sec_tx": "language", "sec_impact": "language"}
fig, axs = plt.subplots(1, 3, figsize=(15, 3.8))
colors = {"vision": "#1f77b4", "indicators": "#2ca02c", "language": "#9467bd"}
axs[0].bar(GW.index, GW.to_numpy(), color=[colors[GROUP[n]] for n in GW.index])
axs[0].set_title("mean gate weight per block (valid rows)"); axs[0].tick_params(axis="x", rotation=45)
share = GW.groupby(GROUP).sum()
axs[1].pie(share.to_numpy(), labels=[f"{k}\n{v:.0%}" for k, v in share.items()], colors=[colors[k] for k in share.index],
           startangle=90, wedgeprops={"width": .45})
axs[1].set_title("gate share by modality")
h = HIST2[("MM", SEEDS[0])]
for c, col in (("train CE", "#1f77b4"), ("inner CE (early stop)", "#ff7f0e"), ("valid CE (shown only)", "#2ca02c")):
    axs[2].plot(h.epoch, h[c], marker="o", ms=3, label=c, color=col)
axs[2].set_title(f"vision + NLP network, seed {SEEDS[0]}: loss curves"); axs[2].set_xlabel("epoch")
axs[2].legend(fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.16), ncol=3, frameon=False)
plt.tight_layout(); plt.show()
print("Gate weights are a model diagnostic, not a causal explanation: they show what the network reads, not why prices move.")

fig, axs = plt.subplots(1, 3, figsize=(13, 3.8))
for ax, hz in zip(axs, ("1D", "5D", "20D")):
    j = H6.index(hz); ok = YE_[IXT, j] >= 0
    cm = confusion_matrix(YE_[IXT[ok], j], PV["MM"][IXT[ok], j].argmax(1), labels=[0, 1, 2], normalize="true")
    ax.imshow(cm, cmap="Blues", vmin=0, vmax=1)
    for a in range(3):
        for b in range(3):
            ax.text(b, a, f"{cm[a, b]:.2f}", ha="center", va="center", color="white" if cm[a, b] > .55 else "black")
    ax.set_xticks(range(3), CLASSES); ax.set_yticks(range(3), CLASSES)
    ax.set_title(f"vision + NLP, {hz}, test (row-normalised)"); ax.set_xlabel("predicted"); ax.set_ylabel("true")
plt.tight_layout(); plt.show()

In [ ]:
# the documented E28 run (package code, the PC): every configuration, the decision and the test report
E28_DIR = REPO_DIR / "experiments" / "20261113T035556_e28_vision_nlp_fusion_b1f899"
if (E28_DIR / "metrics.json").exists():
    M28 = json.loads((E28_DIR / "metrics.json").read_text(encoding="utf-8"))
    res = M28["results"]
    T28 = pd.DataFrame({k: {"blocks": ", ".join(v["modalities"]), "valid mean log loss": v["valid_mean_log_loss"],
                            "test mean log loss": v["test_mean_log_loss"]} for k, v in res.items()}).T
    display(Markdown(f"**Documented run: decision {M28['decision']}** (rule: {M28['decision_rule']})"))
    display(T28.style.format({"valid mean log loss": "{:.5f}", "test mean log loss": "{:.5f}"}))
    D = pd.DataFrame({**{f"valid: {k}": v for k, v in M28["validation"].items()},
                      **{f"valid (reported): {k}": v for k, v in M28["reported_validation"].items()},
                      **{f"test: {k}": v for k, v in M28["test_after_decision"].items()}}).T
    D = D.assign(low=D.ci95.map(lambda c: c[0]), high=D.ci95.map(lambda c: c[1]))[["estimate", "low", "high"]]
    E28_D = D                                                    # kept for the report (section 20)
    display(D.style.format("{:+.5f}"))
    RIC = pd.DataFrame({k: {"BASE": v["BASE"]["estimate"], "MM": v["MM"]["estimate"],
                            "MM - BASE": v["MM_minus_BASE"]["estimate"], "low": v["MM_minus_BASE"]["ci95"][0],
                            "high": v["MM_minus_BASE"]["ci95"][1]} for k, v in M28["rank_ic"].items()}).T
    display(Markdown("**Daily cross-sectional rank IC of P(up) - P(down)** (reported; it decides nothing)"))
    display(RIC.style.format("{:+.4f}"))
else:
    print("the documented E28 run is not in this copy of the repository")

## 9. Optimisation: what "reaching the global minimum" means for each model

* **Logistic regression (section 6):** certified. Strict convexity, five starts to the same point, zero gradient, positive
  definite Hessian.
* **LightGBM:** greedy tree growth has no global optimum to reach; the right question is when to stop, and the
  held-out curve (section 7) answers it.
* **Neural network:** the loss is non-convex and no certificate exists. What matters in practice is that the solutions
  found are equally good on held-out data, so the choice of minimum does not change the conclusions. The cell shows
  (a) the spread of held-out loss across seeds, (b) the loss along the straight line between two seeds' weights
  (a barrier means separate basins; equal endpoints mean equally good ones), and (c) the 2-D loss landscape around the
  E06 optimum saved by the project (a smooth bowl means the optimiser sits at its bottom).

Further tuning on the same validation years would lower the validation loss and raise the risk of overfitting it:
`docs/TRIAL_LEDGER.md` already counts up to 665 selection trials on 2017-2018. A real improvement has to show up on
data no choice has seen: the forward test (section 18) and the sealed period.

In [ ]:
seed_ce = {s: eval_ce(FUSION[s], to_dev(VAL)) for s in SEEDS}
print("valid cross-entropy by seed:", {s: round(v, 5) for s, v in seed_ce.items()},
      "| spread", f"{np.ptp(list(seed_ce.values())):.5f}")
if len(SEEDS) >= 2:
    a, b = FUSION[SEEDS[0]].state_dict(), FUSION[SEEDS[1]].state_dict()
    probe = GatedFusion({n: blocks[n].shape[1] for n in NAMES}).to(DEVICE)
    va_small = to_dev(VAL[:15000])
    path = []
    for t in np.linspace(0, 1, 11):
        probe.load_state_dict({k: (1 - t) * a[k] + t * b[k] if a[k].is_floating_point() else a[k] for k in a})
        path.append(eval_ce(probe, va_small))
    fig, ax = plt.subplots(figsize=(6, 3)); ax.plot(np.linspace(0, 1, 11), path, marker="o")
    ax.set_title(f"valid CE on the line from seed {SEEDS[0]} (0) to seed {SEEDS[1]} (1)"); ax.set_xlabel("t")
    plt.tight_layout(); plt.show()
    print(f"barrier height above the mean of the endpoints: {max(path) - (path[0] + path[-1]) / 2:+.5f}")
L = np.load(NB / "predictions" / "e06__landscape_2d.npy")
e06m = REPO_DIR / "experiments" / "20261106T005012_e06_optimization_fa42d8" / "metrics.json"
grid = (np.asarray(json.loads(e06m.read_text(encoding="utf-8"))["landscape"]["grid"]) if e06m.exists()
        else np.linspace(-1, 1, L.shape[0]))
fig, ax = plt.subplots(figsize=(5.2, 4.2))
cs_ = ax.contourf(grid, grid, L.T, levels=24, cmap="viridis")
ax.plot(0, 0, "r*", ms=12, label="E06 optimum"); plt.colorbar(cs_, ax=ax, label="valid log loss")
ax.set_title("E06: loss landscape around the trained network\n(two random filter-normalised directions)")
ax.legend(fontsize=8); plt.tight_layout(); plt.show()
print(f"landscape: centre {L[L.shape[0] // 2, L.shape[1] // 2]:.5f}, grid minimum {L.min():.5f}")

## 10. Fine-tuning a pretrained vision network on chart images (GPU)

The fusion network reads **frozen** ImageNet features of the charts. Fine-tuning asks whether adapting the network to
charts helps. The cell renders daily charts with the project's canonical renderer **r1** (TriFrame's binary OHLC
bars: open tick left, high-low line centre, close tick right, with a volume panel; 126 daily bars; log price scaled to
the window, so the chart shows shape, not the volatility level), then fine-tunes an ImageNet ConvNeXt-Tiny on the 5D
label: the stem and first two stages stay frozen, the last two stages and a new 3-class head are trained. Training
rows come from the train block, early stopping uses the inner block, and the result is scored on the validation years.

TriFrame fine-tuned the same network on 500 stocks and found the image gain close to zero; E02 confirms it for NEXUS
(the chart-fine-tuned embeddings are compared in section 14). Expect the same here: the point is the honest test.

In [ ]:
FT = {"run": BARS_DIR.exists(), "n_train": 600 if QUICK else (40_000 if DEVICE == "cuda" else 6_000),
      "n_inner": 200 if QUICK else (6_000 if DEVICE == "cuda" else 1_500),
      "n_valid": 400 if QUICK else (12_000 if DEVICE == "cuda" else 3_000),
      "epochs": 1 if QUICK else (4 if DEVICE == "cuda" else 2), "batch": 128 if DEVICE == "cuda" else 32}
RENDER_H, PRICE_H, GAP, VOL_H = 128, 100, 2, 26

def render(o, h, l, c, v, n_bars):
    # TriFrame canonical renderer r1, copied unchanged (nexus_alpha.cv.render.render)
    m = len(o); W = 3 * n_bars
    img = np.zeros((RENDER_H, W), np.uint8)
    if m == 0:
        return img, 0
    o, h, l, c, v = (np.asarray(a, float) for a in (o, h, l, c, v))
    lo, hi = np.log(l.min()), np.log(h.max()); span = hi - lo
    if span > 0:
        f = lambda a: np.rint((np.log(a) - lo) / span * (PRICE_H - 1)).astype(int)
        yo, yh, yl, yc = f(o), f(h), f(l), f(c)
    else:
        yo = yh = yl = yc = np.full(m, PRICE_H // 2)
    x0 = 3 * (n_bars - m) + 3 * np.arange(m)
    rows = np.arange(PRICE_H)[:, None]
    price = np.zeros((PRICE_H, W), np.uint8)
    price[:, x0 + 1] = ((rows >= yl[None, :]) & (rows <= yh[None, :])).astype(np.uint8)
    price[yo, x0] = 1
    price[yc, x0 + 2] = 1
    img[:PRICE_H] = price[::-1]
    vmax = v.max()
    if vmax > 0:
        hv = np.rint(v / vmax * (VOL_H - 1)).astype(int) + (v > 0)
        vol = np.zeros((VOL_H, W), np.uint8)
        vol[:, x0 + 1] = (np.arange(VOL_H)[:, None] < hv[None, :]).astype(np.uint8)
        img[PRICE_H + GAP:] = vol[::-1]
    return img, m

def render_rows(ix, bars_cache):
    out, keep = [], []
    for i in ix:
        t, a = FR.ticker.iat[i], FR.asof_date.iat[i]
        d = bars_cache.get(t)
        if d is None:
            continue
        k = int(np.searchsorted(d.date.to_numpy(), a.to_datetime64(), side="right"))
        w = d.iloc[max(0, k - 126):k]
        if len(w) < 51 or w.date.iat[-1] != a:
            continue
        im, _ = render(w.open.to_numpy(), w.high.to_numpy(), w.low.to_numpy(), w.close.to_numpy(), w.volume.to_numpy(), 126)
        out.append(np.packbits(im, axis=1)); keep.append(i)
    return np.stack(out), np.array(keep)

if FT["run"]:
    import timm
    j5 = H6.index("5D")
    pick = lambda b, n: sample(masks[b] & (Y[:, j5] >= 0) & (FR.n_valid_D.to_numpy() >= 51), n)
    ix_tr, ix_in, ix_va = pick("train", FT["n_train"]), pick("inner", FT["n_inner"]), pick("valid", FT["n_valid"])
    need = sorted(set(FR.ticker.iloc[np.concatenate([ix_tr, ix_in, ix_va])]))
    t0 = time.time()
    bars_cache = {t: read_daily(t) for t in need if (BARS_DIR / f"{t}.parquet").exists()}
    IMG = {k: render_rows(ix, bars_cache) for k, ix in (("train", ix_tr), ("inner", ix_in), ("valid", ix_va))}
    print({k: len(v[1]) for k, v in IMG.items()}, f"charts rendered in {time.time() - t0:.0f}s")
    fig, axs = plt.subplots(1, 3, figsize=(13, 2.2))
    for ax, k in zip(axs, range(3)):
        im = np.unpackbits(IMG["train"][0][k], axis=1)[:, :378]
        ax.imshow(1 - im, cmap="gray", aspect="auto"); ax.set_axis_off()
        ax.set_title(f"{FR.ticker.iat[IMG['train'][1][k]]} {FR.asof_date.iat[IMG['train'][1][k]].date()} "
                     f"-> {CLASSES[Y[IMG['train'][1][k], j5]]} (5D)", fontsize=9)
    plt.tight_layout(); plt.show()

    MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1); STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
    def batch_tensor(packed):
        x = np.unpackbits(packed, axis=2)[:, :, :384].astype(np.float32)          # 378 columns, padded to 384
        x = torch.from_numpy(1 - x)[:, None].repeat(1, 3, 1, 1)                     # white background, dark bars
        return ((x - MEAN) / STD).to(DEVICE, non_blocking=True)
    net = timm.create_model("convnext_tiny.in12k_ft_in1k", pretrained=True, num_classes=3).to(DEVICE)
    for n_, p_ in net.named_parameters():
        p_.requires_grad = not (n_.startswith("stem") or n_.startswith("stages.0") or n_.startswith("stages.1"))
    opt = torch.optim.AdamW([p_ for p_ in net.parameters() if p_.requires_grad], lr=2e-4, weight_decay=0.05)
    def amp():
        return torch.autocast("cuda", dtype=torch.bfloat16) if DEVICE == "cuda" else torch.autocast("cpu", enabled=False)
    def run_eval(split):
        net.eval(); pk, ix = IMG[split]; out = []
        with torch.no_grad(), amp():
            for i in range(0, len(ix), 256):
                out.append(net(batch_tensor(pk[i:i + 256])).float().softmax(-1).cpu().numpy())
        p = np.concatenate(out); y = Y[ix, j5]
        return log_loss(y, np.clip(p, 1e-7, 1), labels=[0, 1, 2]), p
    FT_HIST, best, best_state = [], 1e9, None
    pk_tr, ix_tr2 = IMG["train"]; ytr_t = torch.from_numpy(Y[ix_tr2, j5])
    for ep in range(FT["epochs"]):
        net.train(); perm = np.random.default_rng(ep).permutation(len(ix_tr2)); run, nb = 0.0, 0
        for i in range(0, len(perm), FT["batch"]):
            b = perm[i:i + FT["batch"]]
            with amp():
                loss = F.cross_entropy(net(batch_tensor(pk_tr[b])).float(), ytr_t[b].to(DEVICE))
            opt.zero_grad(); loss.backward(); opt.step(); run += loss.item(); nb += 1
        ll_in, _ = run_eval("inner"); ll_va, _ = run_eval("valid")
        FT_HIST.append({"epoch": ep + 1, "train CE": run / nb, "inner CE (early stop)": ll_in, "valid CE (shown only)": ll_va})
        print(FT_HIST[-1], f"({time.time() - t0:.0f}s)")
        if ll_in < best:
            best, best_state = ll_in, {k: v.detach().clone() for k, v in net.state_dict().items()}
    net.load_state_dict(best_state)
    ll_va, p_va = run_eval("valid")
    y_va = Y[IMG["valid"][1], j5]
    prior5 = np.bincount(Y[TR, j5][Y[TR, j5] >= 0], minlength=3) / (Y[TR, j5] >= 0).sum()
    ll_prior = log_loss(y_va, np.tile(prior5, (len(y_va), 1)), labels=[0, 1, 2])
    FT_RESULT = {"valid rows": len(y_va), "fine-tuned CNN log loss": ll_va, "prior log loss": ll_prior,
                 "gain vs prior (positive = better)": ll_prior - ll_va,
                 "up-vs-down AUROC": roc_auc_score(y_va[y_va != 1] == 2, (p_va[:, 2] / (p_va[:, 0] + p_va[:, 2]))[y_va != 1])}
    h = pd.DataFrame(FT_HIST)
    fig, ax = plt.subplots(figsize=(6, 3.2))
    for c, col in (("train CE", "#1f77b4"), ("inner CE (early stop)", "#ff7f0e"), ("valid CE (shown only)", "#2ca02c")):
        ax.plot(h.epoch, h[c], marker="o", label=c, color=col)
    ax.axhline(ll_prior, color="k", ls="--", lw=1, label="prior (valid)")
    ax.set_title("ConvNeXt-Tiny fine-tuned on daily charts, 5D label"); ax.legend(fontsize=8); plt.tight_layout(); plt.show()
    display(pd.Series(FT_RESULT).to_frame("value"))
else:
    print("Daily bars not found: the fine-tuning section is skipped.")

## 11. Calibration

A probability is useful only if 30% means 30%. Each horizon gets one temperature T (logits divided by T) fitted on the
**calibration** block (2019) by minimising its log loss, then applied unchanged to validation and test. The reliability
diagram plots the predicted probability of the top class against how often it was right; ECE is the average gap,
weighted by the number of rows in each bin.

In [ ]:
POS = {i: k for k, i in enumerate(EVI)}                      # row index -> position in EVAL arrays
def rows_of(block):
    return np.array([POS[i] for i in np.flatnonzero(masks[block])])
P_CAL, P_VAL, P_TEST = rows_of("calib"), rows_of("valid"), rows_of("test")
YE = Y[EVI]

def fit_temperature(p, y):
    lg = np.log(np.clip(p, 1e-7, 1))
    f = lambda T: log_loss(y, np.exp(lg / T) / np.exp(lg / T).sum(1, keepdims=True), labels=[0, 1, 2])
    return optimize.minimize_scalar(f, bounds=(0.3, 5.0), method="bounded").x

def apply_temperature(p, T):
    lg = np.log(np.clip(p, 1e-7, 1)) / T
    e = np.exp(lg - lg.max(-1, keepdims=True)); return e / e.sum(-1, keepdims=True)

def ece(p, y, bins=15):
    conf, pred = p.max(1), p.argmax(1)
    edges = np.linspace(0, 1, bins + 1); tot = 0.0
    for a, b in zip(edges[:-1], edges[1:]):
        s = (conf > a) & (conf <= b)
        if s.any():
            tot += s.mean() * abs((pred[s] == y[s]).mean() - conf[s].mean())
    return tot

cal = PRED["fusion"].copy(); TEMPS = {}
for j, h in enumerate(H6):
    ok = YE[P_CAL, j] >= 0
    TEMPS[h] = fit_temperature(PRED["fusion"][P_CAL[ok], j], YE[P_CAL[ok], j])
    cal[:, j] = apply_temperature(PRED["fusion"][:, j], TEMPS[h])
PRED["fusion_cal"] = cal
print("temperatures (above 1 = the raw network was over-confident):", {h: round(float(t), 3) for h, t in TEMPS.items()})

fig, axs = plt.subplots(1, 3, figsize=(13, 3.8))
for ax, h in zip(axs, ["1D", "5D", "20D"]):
    j = H6.index(h); ok = YE[P_TEST, j] >= 0; y = YE[P_TEST[ok], j]
    for name, col in (("fusion", "#d62728"), ("fusion_cal", "#1f77b4"), ("prior", "#9aa5b1")):
        p = PRED[name][P_TEST[ok], j]; conf, hit = p.max(1), (p.argmax(1) == y)
        q = pd.qcut(conf, 10, duplicates="drop")
        g = pd.DataFrame({"c": conf, "h": hit}).groupby(q, observed=True).mean()
        ax.plot(g.c, g.h, marker="o", label=f"{name} (ECE {ece(p, y):.3f})", color=col)
    lim = [0.3, 0.7]; ax.plot(lim, lim, "k--", lw=.8); ax.set_xlim(lim); ax.set_ylim(lim)
    ax.set_title(f"reliability on test, {h}"); ax.set_xlabel("predicted top-class probability"); ax.set_ylabel("observed")
    ax.legend(fontsize=7)
plt.tight_layout(); plt.show()

## 12. Evaluation: every metric, every horizon

Models trained in this notebook (prior, logistic, LightGBM, fusion with and without calibration) and the frozen models'
saved predictions on the same rows:
* **E08 = v1.1**, the frozen production model (a blend: 60% fusion over five seeds, 30% logistic, 10% LightGBM). It was
  refitted on 2003-2018 before freezing, so **its validation rows are in-sample**: judge it on test.
* **E03 gated fusion**, the documented run of section 8's architecture (fitted on 2003-2015).

Metrics: **log loss** (the primary endpoint; lower is better), **Brier score**, **accuracy**, **balanced accuracy**,
**macro-F1**, **AUROC** (one-vs-rest, macro), **up-vs-down AUROC** (direction skill among moves), **sideways AUROC**
(volatility-regime skill), **ECE**, and **rank IC** (per-date Spearman correlation of P(up) - P(down) with the
realised return, averaged over dates).

**Reading the confusion matrices.** They count the most likely class. SIDEWAYS is the largest class, and the
probabilities stay close to the class shares, so the most likely class is nearly always SIDEWAYS at short horizons.
A matrix like that says the forecasts are cautious, not that they are useless. The information is in how far the
probabilities move from the prior, which is what log loss, AUROC and rank IC measure.

In [ ]:
ROWS = pd.read_parquet(NB / "predictions" / "rows_eval.parquet")
assert (ROWS.sample_id.to_numpy() == FR.sample_id.to_numpy()[EVI]).all(), "saved predictions are not aligned"
def saved(name):
    return np.load(NB / "predictions" / f"{name}.npy").astype(np.float64)
for k, f in (("v1.1 (E08 frozen)", "e08__final_deployed_calibrated"), ("E03 gated (saved)", "e03__variant_gated"),
             ("E01 LightGBM (saved)", "e01__lgbm_struct_x33"), ("E06 seed ensemble (saved)", "e06__best_seed_ensemble")):
    if (NB / "predictions" / f"{f}.npy").exists():
        PRED[k] = saved(f)

def rank_ic(score, ret, dates, min_n=10):
    # per-date Spearman correlation, vectorised: Pearson correlation of within-date ranks
    df = pd.DataFrame({"s": score, "r": ret, "d": dates}).dropna()
    g = df.groupby("d")
    a = g.s.rank() - g.s.rank().groupby(df.d).transform("mean")
    b = g.r.rank() - g.r.rank().groupby(df.d).transform("mean")
    num = (a * b).groupby(df.d).sum()
    den = np.sqrt((a ** 2).groupby(df.d).sum() * (b ** 2).groupby(df.d).sum())
    ics = (num / den)[g.size() >= min_n].replace([np.inf, -np.inf], np.nan).dropna()
    return float(ics.mean()), ics

def metrics(p, y, ret=None, dates=None):
    ok = y >= 0; p, y = np.clip(p[ok], 1e-7, 1), y[ok]; p = p / p.sum(1, keepdims=True)
    yo = np.eye(3)[y]
    out = {"n": int(ok.sum()), "log loss": log_loss(y, p, labels=[0, 1, 2]), "Brier": float(((p - yo) ** 2).sum(1).mean()),
           "accuracy": accuracy_score(y, p.argmax(1)), "balanced accuracy": balanced_accuracy_score(y, p.argmax(1)),
           "macro-F1": f1_score(y, p.argmax(1), average="macro"),
           "AUROC (macro OvR)": roc_auc_score(yo, p, average="macro", multi_class="ovr"), "ECE": ece(p, y)}
    mv = y != 1
    out["up-vs-down AUROC"] = roc_auc_score(y[mv] == 2, (p[mv, 2] / (p[mv, 0] + p[mv, 2])))
    out["sideways AUROC"] = roc_auc_score(y == 1, p[:, 1])
    if ret is not None:
        out["rank IC"] = rank_ic(p[:, 2] - p[:, 0], ret[ok], dates[ok])[0]
    return out

DATES = FR.asof_date.to_numpy()[EVI]; RETE = RET[EVI]
TABLE = []
for name, P in PRED.items():
    for split, ix in (("valid", P_VAL), ("test", P_TEST)):
        for j, h in enumerate(H6):
            TABLE.append({"model": name, "split": split, "horizon": h,
                          **metrics(P[ix, j], YE[ix, j], RETE[ix, j], DATES[ix])})
TABLE = pd.DataFrame(TABLE)
# A-90 (audit 9): v1.1 was refitted on 2003-2018, so its 2017-18 rows are in-sample; say so in the table itself
IN_SAMPLE_VALID = {"v1.1 (E08 frozen)"}
TABLE["note"] = np.where(TABLE.model.isin(IN_SAMPLE_VALID) & (TABLE.split == "valid"),
                         "IN-SAMPLE (refit 2003-2018): not comparable", "")
LABEL = {"prior": "class prior", "logistic": "logistic (trained here)", "lgbm": "LightGBM (trained here)",
         "fusion": "fusion network (trained here)", "fusion_cal": "fusion network, calibrated (trained here)",
         "fusion_mm": "vision + NLP fusion network (trained here)"}
mean_ll = TABLE.pivot_table(index="model", columns="split", values="log loss", aggfunc="mean").sort_values("test")
mean_ll["note"] = ["valid column in-sample (refit 2003-2018)" if i in IN_SAMPLE_VALID else "" for i in mean_ll.index]
mean_ll.index = [LABEL.get(i, i) for i in mean_ll.index]
display(Markdown("**Mean log loss over the six horizons (lower is better)**"))
display(mean_ll.style.format({"test": "{:.5f}", "valid": "{:.5f}"}))
show = TABLE[TABLE.split == "test"].set_index(["model", "horizon"]).drop(columns=["split", "note"])
display(Markdown("**All metrics on test (2020-2022)**")); display(show.style.format(precision=4))

In [ ]:
# confusion matrices of the frozen v1.1 model on test, by horizon (rows: true class; columns: predicted; row-normalised)
name = "v1.1 (E08 frozen)" if "v1.1 (E08 frozen)" in PRED else "fusion_cal"
fig, axs = plt.subplots(1, 6, figsize=(18, 3.3))
for ax, (j, h) in zip(axs, enumerate(H6)):
    ok = YE[P_TEST, j] >= 0
    cm = confusion_matrix(YE[P_TEST[ok], j], PRED[name][P_TEST[ok], j].argmax(1), labels=[0, 1, 2], normalize="true")
    ax.imshow(cm, cmap="Blues", vmin=0, vmax=1)
    for a in range(3):
        for b in range(3):
            ax.text(b, a, f"{cm[a, b]:.2f}", ha="center", va="center", color="white" if cm[a, b] > .5 else "black", fontsize=9)
    ax.set_xticks(range(3), ["D", "S", "U"]); ax.set_yticks(range(3), ["D", "S", "U"]); ax.set_title(h); ax.grid(False)
fig.suptitle(f"{name}: confusion matrices on test (row-normalised)", y=1.03); plt.tight_layout(); plt.show()

# ROC curves: direction (up vs down) and volatility regime (sideways vs moves), 5D, test
j = H6.index("5D"); ok = YE[P_TEST, j] >= 0; y = YE[P_TEST[ok], j]
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 4))
for nm, col in (("logistic", CLR["logistic"]), ("lgbm", CLR["lgbm"]), ("fusion_cal", CLR["fusion"]), (name, CLR["v1.1"])):
    if nm not in PRED:
        continue
    p = PRED[nm][P_TEST[ok], j]; mv = y != 1
    fpr, tpr, _ = roc_curve(y[mv] == 2, p[mv, 2] / (p[mv, 0] + p[mv, 2]))
    a1.plot(fpr, tpr, color=col, label=f"{nm} {roc_auc_score(y[mv] == 2, p[mv, 2] / (p[mv, 0] + p[mv, 2])):.3f}")
    fpr, tpr, _ = roc_curve(y == 1, p[:, 1])
    a2.plot(fpr, tpr, color=col, label=f"{nm} {roc_auc_score(y == 1, p[:, 1]):.3f}")
for ax, t in ((a1, "5D direction: up vs down (moves only)"), (a2, "5D volatility regime: sideways vs moves")):
    ax.plot([0, 1], [0, 1], "k--", lw=.8); ax.set_title(t); ax.legend(fontsize=8, title="AUROC"); ax.set_xlabel("false positive rate")
plt.tight_layout(); plt.show()

## 13. Statistics: is any gain real?

* **Month-block bootstrap.** The log-loss gain of a model over the prior, resampling whole calendar months (rows in
  the same month share market moves, so resampling single rows would overstate certainty). 95% intervals and
  one-sided p-values.
* **Holm correction** across the six horizons (one family).
* **Placebo.** The model's forecasts shuffled across stocks within each date: market-level information survives, the
  stock-specific link does not. A real stock-level signal must beat its own placebo.
* **Rank IC** with a month-block interval: the cross-sectional ranking, the part of the signal the project validated.

In [ ]:
def month_block_boot(delta, dates, B=1000, seed=SEED):
    months = pd.DatetimeIndex(dates).to_period("M")
    g = pd.DataFrame({"d": delta, "m": months}).groupby("m").d.agg(["sum", "count"])
    s, c = g["sum"].to_numpy(), g["count"].to_numpy()
    r = np.random.default_rng(seed); k = len(g)
    bs = np.array([(s[ix].sum() / c[ix].sum()) for ix in (r.integers(0, k, k) for _ in range(B))])
    return float(delta.mean()), np.percentile(bs, [2.5, 97.5]), float((bs <= 0).mean())

def holm(p):
    p = np.asarray(p); o = np.argsort(p); adj = np.empty_like(p, dtype=float); run = 0.0
    for r, i in enumerate(o):
        run = max(run, (len(p) - r) * p[i]); adj[i] = min(1.0, run)
    return adj

def row_ll(P, y):
    return -np.log(np.clip(P[np.arange(len(y)), y], 1e-7, 1))

def placebo(P, dates, seed=SEED):
    # shuffle forecasts across stocks within each date
    r = np.random.default_rng(seed); out = P.copy()
    for d, ix in pd.Series(np.arange(len(dates))).groupby(dates).groups.items():
        ix = np.asarray(ix); out[ix] = P[r.permutation(ix)]
    return out

name = "v1.1 (E08 frozen)" if "v1.1 (E08 frozen)" in PRED else "fusion_cal"
rows = []
for j, h in enumerate(H6):
    ok = YE[P_TEST, j] >= 0; ix = P_TEST[ok]; y = YE[ix, j]; d = DATES[ix]
    base = row_ll(PRED["prior"][ix, j], y)
    real = base - row_ll(PRED[name][ix, j], y)
    pl = base - row_ll(placebo(PRED[name][ix, j], d), y)
    est, ci, p = month_block_boot(real, d)
    pest, pci, _ = month_block_boot(pl, d)
    ic, ics = rank_ic(PRED[name][ix, j, 2] - PRED[name][ix, j, 0], RETE[ix, j], d)
    icm = ics.groupby(pd.DatetimeIndex(ics.index).to_period("M")).mean()
    bs = [np.random.default_rng(b).choice(icm.to_numpy(), len(icm)).mean() for b in range(1000)]
    rows.append({"horizon": h, "LL gain vs prior": est, "95% CI": f"[{ci[0]:+.4f}, {ci[1]:+.4f}]", "p (one-sided)": p,
                 "placebo gain": pest, "real minus placebo": est - pest,
                 "rank IC": ic, "rank IC 95% CI": f"[{np.percentile(bs, 2.5):+.4f}, {np.percentile(bs, 97.5):+.4f}]"})
ST = pd.DataFrame(rows).set_index("horizon")
ST["p Holm"] = holm(ST["p (one-sided)"].to_numpy())
display(Markdown(f"**{name} on test (2020-2022): gain over the class prior, placebo and rank IC**"))
display(ST.style.format({"LL gain vs prior": "{:+.5f}", "placebo gain": "{:+.5f}", "real minus placebo": "{:+.5f}",
                         "p (one-sided)": "{:.3f}", "p Holm": "{:.3f}", "rank IC": "{:+.4f}"}))

## 13A. Modality scorecard: vision vs NLP vs both (descriptive)

Which inputs carry the forecastable information: the charts (computer vision), the text (NLP), the indicators, or a
combination? The same gated fusion network as section 8 (same architecture, optimiser, early stopping on the inner
block) is trained on each set of blocks, so the only difference between two rows of the table is the input.

| Configuration | Blocks |
|---|---|
| vision only | weekly, daily and hourly chart embeddings |
| NLP only | news (TF-IDF, tone, timing), SEC (TF-IDF, tone, items, timing), FinBERT news, ChronoBERT filings, the SEC price-impact expert |
| indicators only | the 80 indicator features |
| vision + NLP | charts and every language block, no indicators |
| vision + indicators | charts and indicators, no language |
| indicators + NLP | indicators and language, no charts |
| all | charts, indicators and language (section 8C's MM network) |

**Seeds.** Each configuration is an average of **five seeds** in full mode (7, 11, 23, 42, 101: the seeds of E06 and
E08), so the table also shows how much each number moves from one seed to the next (audit finding A-84: the seed spread
is as large as many of the differences tested). Two configurations whose gap is smaller than their seed spread are not
distinguishable, whatever the interval says.

**Metrics, per configuration, on validation (2017-18) and test (2020-22):** log loss and its gain over the class prior,
Brier score, accuracy, balanced accuracy, macro-F1, AUROC, ECE (calibration error), up-vs-down AUROC (direction),
sideways AUROC (whether the stock moves), rank IC (cross-sectional ranking). The quantile heads are scored too: how often
the realised return lands inside the predicted 10-90% range (target 80%) and below the 10% quantile (target 10%), the
natural distance for a stop-loss.

**A fair test for text.** Language exists only where there was news or a filing before the decision time, and the news
source covers far fewer stocks in 2020-22 than in 2017-18 (the coverage table below). The scorecard is shown on all rows
and on the rows with any text. On rows without text an NLP-only network has no input at all: its output is a constant,
so its quantile coverage there says nothing about language (shown separately).

**Increments** are paired on the same rows: the per-row log-loss difference, averaged over the six horizons, with a
month-block bootstrap 95% interval (positive = the added input helps), a one-sided bootstrap p-value (the share of
resamples at or below zero), its Holm adjustment over the eight comparisons of each row set, and the range of the same
increment computed seed by seed.

**This section is descriptive.** Nothing is selected from it: E28's decision (section 8C) is the recorded one, and the
validation years have already served the trial ledger's selection trials (`docs/TRIAL_LEDGER.md`).

In [ ]:
VISION_B, LANG_B, IND_B = ["img_W", "img_D", "img_H"], ["news", "sec"] + list(NEWB), ["num"]
ALLK = "all (vision + indicators + NLP)"
MOD_CFG = {"vision only": VISION_B, "NLP only": LANG_B, "indicators only": IND_B, "vision + NLP": VISION_B + LANG_B,
           "vision + indicators": VISION_B + IND_B, "indicators + NLP": IND_B + LANG_B}
SC_ORDER = ["prior"] + list(MOD_CFG) + [ALLK]
SEEDS13 = list(SEEDS) if QUICK else [7, 11, 23, 42, 101]

@torch.no_grad()
def predict_quant(model, ix, bs=8192, B=None):
    model.eval(); xs, m, _, _ = to_dev(ix, B); out_ = []
    for i in range(0, len(m), bs):
        _, qu, _ = model({k: v[i:i + bs] for k, v in xs.items()}, m[i:i + bs])
        out_.append(qu.float().cpu().numpy())
    return np.sort(np.concatenate(out_), axis=-1)            # (n, 6, 3): q10, q50, q90 of the vol-scaled return

def row_ll_all(P):
    # per-row, per-horizon log loss on every evaluation row (NaN where the row has no label)
    L = np.full((len(EVI), 6), np.nan, np.float32)
    for j in range(6):
        y = YE[:, j]; ok = y >= 0
        L[ok, j] = -np.log(np.clip(P[np.flatnonzero(ok), j, y[ok]], 1e-7, 1))
    return L

QV, SC_EPOCHS, LLS = {}, {}, {}
t0 = time.time()
for name, names_ in MOD_CFG.items():
    Bk = block_set(names_)
    ms_, hs_ = zip(*[train_fusion(s, epochs=2 if QUICK else 30, B=Bk) for s in SEEDS13])
    ps_ = [predict_probs(m_, EVI, B=Bk) for m_ in ms_]
    PV[name] = np.mean(ps_, 0)
    QV[name] = np.mean([predict_quant(m_, EVI, B=Bk) for m_ in ms_], 0)
    LLS[name] = [row_ll_all(p_) for p_ in ps_]
    SC_EPOCHS[name] = [len(h_) for h_ in hs_]
    print(f"{name}: {len(SEEDS13)} seeds, epochs {SC_EPOCHS[name]} ({time.time() - t0:.0f} s)")
# "all" = section 8C's MM network; its extra seeds are trained here with the same settings
MM_MODELS = {s: FUS2[("MM", s)] for s in SEEDS if ("MM", s) in FUS2}
for s in SEEDS13:
    if s not in MM_MODELS:
        MM_MODELS[s], h_ = train_fusion(s, epochs=2 if QUICK else 30, B=SETS["MM"])
        HIST2[("MM", s)] = h_
ps_ = [predict_probs(MM_MODELS[s], EVI, B=SETS["MM"]) for s in SEEDS13]
PV[ALLK] = np.mean(ps_, 0)
LLS[ALLK] = [row_ll_all(p_) for p_ in ps_]
QV[ALLK] = np.mean([predict_quant(MM_MODELS[s], EVI, B=SETS["MM"]) for s in SEEDS13], 0)
SC_EPOCHS[ALLK] = [len(HIST2[("MM", s)]) for s in SEEDS13]
PV["prior"] = PRED["prior"]
LLS["prior"] = [row_ll_all(PRED["prior"])]
print(f"all: {len(SEEDS13)} seeds ({time.time() - t0:.0f} s)")

ZE = ZV[EVI]                                                 # realised vol-scaled returns (the quantile heads' target)
HAS_TEXT = np.stack([present2[n] for n in LANG_B], 1).max(1)[EVI] > 0
HAS_NEWS = np.maximum(present2["news"], present2["news_tx"])[EVI] > 0
print(f"rows with any text: valid {HAS_TEXT[P_VAL].mean():.1%}, test {HAS_TEXT[P_TEST].mean():.1%}; "
      f"with company news: valid {HAS_NEWS[P_VAL].mean():.1%}, test {HAS_NEWS[P_TEST].mean():.1%}")

# text coverage by year, every row of the sample (the news source thins out in 2020-22: finding A-99)
yr_all = FR.asof_date.dt.year.to_numpy()
TXT_COV = pd.DataFrame({"rows": pd.Series(1, index=yr_all).groupby(level=0).sum(),
                        "any text": pd.Series(np.stack([present2[n] for n in LANG_B], 1).max(1) > 0, index=yr_all).groupby(level=0).mean(),
                        "company news": pd.Series(np.maximum(present2["news"], present2["news_tx"]) > 0, index=yr_all).groupby(level=0).mean(),
                        "SEC filing in 14 days": pd.Series(present2["sec_impact"] > 0, index=yr_all).groupby(level=0).mean()})
TXT_COV.index.name = "year"
TXT_COV["block"] = FR.groupby(yr_all).block.agg(lambda b: b.fillna("purged").mode().iat[0]).reindex(TXT_COV.index)

SC_PER = {}                                                   # (subset, configuration) -> metrics by horizon

def seed_ll(name, ix):
    # mean six-horizon log loss of each single-seed model on rows ix
    return [float(np.mean([np.nanmean(L[ix, j]) for j in range(6)])) for L in LLS[name]]

def scorecard(ix, label):
    rows = {}
    for name in SC_ORDER:
        P = PV[name]
        per = pd.DataFrame([metrics(P[ix, j], YE[ix, j], RETE[ix, j], DATES[ix]) for j in range(6)], index=H6)
        SC_PER[(label, name)] = per
    base = SC_PER[(label, "prior")]["log loss"].mean()
    for name in SC_ORDER:
        per = SC_PER[(label, name)]
        r = per.drop(columns=["n"]).mean().to_dict()
        r = {"mean log loss": r.pop("log loss"), "gain vs prior": float(base - per["log loss"].mean()), **r}
        r["rank IC 5D"], r["rank IC 20D"] = per.loc["5D", "rank IC"], per.loc["20D", "rank IC"]
        r.pop("rank IC", None)
        sl = seed_ll(name, ix)
        r["seed SD of mean log loss"] = float(np.std(sl, ddof=1)) if len(sl) > 1 else np.nan
        if name in QV:
            q, z = QV[name][ix], ZE[ix]; ok = np.isfinite(z)
            r["q10-q90 coverage (target 0.80)"] = float(((z >= q[..., 0]) & (z <= q[..., 2]))[ok].mean())
            r["below q10 (target 0.10)"] = float((z < q[..., 0])[ok].mean())
        rows[name] = r
    return pd.DataFrame(rows).T

t0 = time.time()
SC_TEST, SC_VALID = scorecard(P_TEST, "test"), scorecard(P_VAL, "valid")
SC_TEST_TEXT = scorecard(P_TEST[HAS_TEXT[P_TEST]], "test, text")
SC_VALID_TEXT = scorecard(P_VAL[HAS_TEXT[P_VAL]], "valid, text")
print(f"scorecards: {time.time() - t0:.0f} s")
FMT = {c: "{:.4f}" for c in SC_TEST.columns} | {"gain vs prior": "{:+.5f}", "rank IC 5D": "{:+.4f}", "rank IC 20D": "{:+.4f}",
                                               "seed SD of mean log loss": "{:.5f}"}
display(Markdown("**Test 2020-22, all rows: every metric averaged over the six horizons** (green = better)"))
display(SC_TEST.style.format(FMT, na_rep="-").background_gradient(cmap="RdYlGn", subset=["gain vs prior"])
        .background_gradient(cmap="RdYlGn", subset=["up-vs-down AUROC", "sideways AUROC"]))
COMPACT = ["mean log loss", "gain vs prior", "seed SD of mean log loss", "up-vs-down AUROC", "sideways AUROC",
           "rank IC 5D", "rank IC 20D", "ECE"]
for title, tab in (("Test 2020-22, rows with news or a filing", SC_TEST_TEXT), ("Validation 2017-18, all rows", SC_VALID),
                   ("Validation 2017-18, rows with news or a filing", SC_VALID_TEXT)):
    display(Markdown(f"**{title}**"))
    display(tab[COMPACT].style.format(FMT, na_rep="-").background_gradient(cmap="RdYlGn", subset=["gain vs prior"]))

SC_SEEDS = pd.DataFrame({name: {"seeds": len(LLS[name]), "epochs per seed (cap 30)": SC_EPOCHS.get(name, "-"),
                                "valid: mean log loss per seed": [round(v, 5) for v in seed_ll(name, P_VAL)],
                                "test: mean log loss per seed": [round(v, 5) for v in seed_ll(name, P_TEST)]}
                         for name in SC_ORDER if name != "prior"}).T
display(Markdown("**Seeds and early stopping per configuration** (every network stops on the inner block before its "
                 "30-epoch cap)")); display(SC_SEEDS)
display(Markdown("**Text coverage by year** (share of stock-dates with each kind of text before the decision time)"))
display(TXT_COV.style.format({"rows": "{:,.0f}", "any text": "{:.1%}", "company news": "{:.1%}",
                              "SEC filing in 14 days": "{:.1%}"}))

# quantile coverage with and without text (on rows without text an NLP-only network sees no input at all)
cov_rows = {}
for name in [c for c in SC_ORDER if c in QV]:
    for split, ix0 in (("valid", P_VAL), ("test", P_TEST)):
        for lab, sel in (("with text", HAS_TEXT[ix0]), ("without text", ~HAS_TEXT[ix0])):
            ix = ix0[sel]; q, z = QV[name][ix], ZE[ix]; ok = np.isfinite(z)
            cov_rows[(name, split, lab)] = {"rows": int(len(ix)),
                                            "q10-q90 coverage": float(((z >= q[..., 0]) & (z <= q[..., 2]))[ok].mean())}
SC_COVER = pd.DataFrame(cov_rows).T.unstack([1, 2])
display(Markdown("**Quantile coverage (target 0.80) with and without text**")); display(SC_COVER.round(3))

In [ ]:
# paired increments: what each input adds on top of the others (per-row log loss, six horizons, month-block bootstrap)
def row_ll6(P, ix):
    L = np.full((len(ix), 6), np.nan)
    for j in range(6):
        y = YE[ix, j]; ok = y >= 0
        L[ok, j] = -np.log(np.clip(P[ix[ok], j, y[ok]], 1e-7, 1))
    return L

def increment(with_, without, ix):
    d = np.nanmean(row_ll6(PV[without], ix) - row_ll6(PV[with_], ix), 1)
    ok = np.isfinite(d)
    est, ci, p = month_block_boot(d[ok], DATES[ix][ok])
    # the same increment seed by seed (seed k of one configuration against seed k of the other)
    Lw, Lo = LLS[with_], LLS[without]
    per_seed = [float(np.nanmean(np.nanmean(Lo[min(k, len(Lo) - 1)][ix] - Lw[k][ix], 1))) for k in range(len(Lw))]
    return {"gain": est, "low": float(ci[0]), "high": float(ci[1]), "p (one-sided)": p,
            "seed range low": min(per_seed), "seed range high": max(per_seed)}

INC = [("vision alone beats the prior", "vision only", "prior"),
       ("NLP alone beats the prior", "NLP only", "prior"),
       ("indicators alone beat the prior", "indicators only", "prior"),
       ("NLP added to vision", "vision + NLP", "vision only"),
       ("vision added to NLP", "vision + NLP", "NLP only"),
       ("NLP added to vision + indicators", ALLK, "vision + indicators"),
       ("vision added to indicators + NLP", ALLK, "indicators + NLP"),
       ("indicators added to vision + NLP", ALLK, "vision + NLP")]
SPL = {"valid": P_VAL, "test": P_TEST, "test, rows with text": P_TEST[HAS_TEXT[P_TEST]]}
SC_INC = pd.DataFrame({(lab, sp): increment(a, b, ix) for lab, a, b in INC for sp, ix in SPL.items()}).T
SC_INC.index.names = ["comparison", "rows"]
SC_INC["p Holm (8 per row set)"] = np.nan
for sp in SPL:                                               # A-97: eight comparisons per row set
    m_ = SC_INC.index.get_level_values("rows") == sp
    SC_INC.loc[m_, "p Holm (8 per row set)"] = holm(SC_INC.loc[m_, "p (one-sided)"].to_numpy(float))
SC_INC["clear (interval and seed range above 0, Holm p < 0.05)"] = ((SC_INC["low"] > 0) & (SC_INC["seed range low"] > 0)
                                                                   & (SC_INC["p Holm (8 per row set)"] < 0.05))
display(Markdown("**What each input adds** (gain in mean log loss per row; positive = better; 95% month-block interval; "
                 "Holm-adjusted p; the seed-by-seed range)"))
display(SC_INC.style.format({"gain": "{:+.5f}", "low": "{:+.5f}", "high": "{:+.5f}", "p (one-sided)": "{:.3f}",
                             "p Holm (8 per row set)": "{:.3f}", "seed range low": "{:+.5f}", "seed range high": "{:+.5f}"})
        .apply(lambda r: ["color: #2ca02c" if r["low"] > 0 else ("color: #d62728" if r["high"] < 0 else "")] * len(r), axis=1))

# stability: the gain over the prior year by year (v1.1's 2017-18 rows are in-sample, so they are left blank)
YRS = pd.DatetimeIndex(DATES).year.to_numpy()
yr_rows = {}
for name in [c for c in SC_ORDER if c != "prior"] + (["v1.1 (E08 frozen)"] if "v1.1 (E08 frozen)" in PRED else []):
    P = PV.get(name, PRED.get(name))
    r = {}
    for y_ in sorted(set(YRS[np.concatenate([P_VAL, P_TEST])])):
        ix = np.concatenate([P_VAL, P_TEST]); ix = ix[YRS[ix] == y_]
        if name == "v1.1 (E08 frozen)" and y_ < 2019:
            r[str(y_)] = np.nan
            continue
        lp = np.mean([np.nanmean(row_ll6(PV["prior"], ix)[:, j]) for j in range(6)])
        lm = np.mean([np.nanmean(row_ll6(P, ix)[:, j]) for j in range(6)])
        r[str(y_)] = float(lp - lm)
    yr_rows[name] = r
SC_YEAR = pd.DataFrame(yr_rows).T
SC_YEAR["years with a gain"] = (SC_YEAR > 0).sum(1).astype(str) + " of " + SC_YEAR.notna().sum(1).astype(str)
display(Markdown("**Gain over the class prior, year by year** (mean six-horizon log loss; v1.1 blank where in-sample)"))
display(SC_YEAR.style.format({c: "{:+.5f}" for c in SC_YEAR.columns if c != "years with a gain"}, na_rep="in-sample"))

fig, axs = plt.subplots(1, 3, figsize=(17, 4.2))
cfgs = [c for c in SC_ORDER if c != "prior"]
xx = np.arange(len(cfgs))
axs[0].bar(xx - .2, SC_VALID.loc[cfgs, "gain vs prior"], .4, label="valid 2017-18", color="#1f77b4",
           yerr=SC_VALID.loc[cfgs, "seed SD of mean log loss"].fillna(0), capsize=2)
axs[0].bar(xx + .2, SC_TEST.loc[cfgs, "gain vs prior"], .4, label="test 2020-22", color="#ff7f0e",
           yerr=SC_TEST.loc[cfgs, "seed SD of mean log loss"].fillna(0), capsize=2)
axs[0].axhline(0, color="k", lw=.8); axs[0].set_xticks(xx, cfgs, rotation=35, ha="right"); axs[0].legend(fontsize=8)
axs[0].set_title("log-loss gain over the class prior (bars: +/- 1 seed SD)")
for ax, col, title in ((axs[1], "up-vs-down AUROC", "direction: up-vs-down AUROC, test"),
                       (axs[2], "sideways AUROC", "does it move: sideways AUROC, test")):
    Mx = pd.DataFrame({c: SC_PER[("test", c)][col] for c in cfgs}).T
    im = ax.imshow(Mx.to_numpy(), cmap="RdYlGn", vmin=0.45 if "up" in col else 0.5, vmax=0.55 if "up" in col else 0.65,
                   aspect="auto")
    ax.set_xticks(range(6), H6); ax.set_yticks(range(len(cfgs)), cfgs); ax.set_title(title); ax.grid(False)
    for a_ in range(len(cfgs)):
        for b_ in range(6):
            ax.text(b_, a_, f"{Mx.iat[a_, b_]:.3f}", ha="center", va="center", fontsize=7)
    plt.colorbar(im, ax=ax, fraction=.03)
plt.tight_layout(); plt.show()

single = SC_TEST.loc[["vision only", "NLP only", "indicators only"], "gain vs prior"]
best_single = single.idxmax()
combo = SC_TEST.loc[ALLK, "gain vs prior"]
vn = SC_INC.loc[("NLP added to vision", "test")]
va_ = SC_INC.loc[("vision added to indicators + NLP", "test")]
SC_VERDICT = (f"On test, the strongest single input is **{best_single}** (gain {single.max():+.5f}); vision alone "
              f"{single['vision only']:+.5f}, NLP alone {single['NLP only']:+.5f}, indicators alone "
              f"{single['indicators only']:+.5f}. Vision + NLP: {SC_TEST.loc['vision + NLP', 'gain vs prior']:+.5f} "
              f"(NLP adds {vn['gain']:+.5f} [{vn['low']:+.5f}, {vn['high']:+.5f}] to vision). All three together: "
              f"{combo:+.5f}; the charts add {va_['gain']:+.5f} [{va_['low']:+.5f}, {va_['high']:+.5f}] once indicators "
              "and language are present. Direction (up-vs-down AUROC) stays near 0.5 for every input; the gains come "
              "from predicting whether a stock moves (sideways AUROC) and from cross-sectional ranking (rank IC). "
              f"Seed-to-seed spread of the mean log loss: up to {SC_TEST['seed SD of mean log loss'].max():.5f} (SD), so "
              "smaller gaps between configurations are not distinguishable.")
display(Markdown(SC_VERDICT))

## 13B. Fundamentals: published scores, and whether they rank returns (E30)

The project already computes 31 point-in-time ratios and 7 multiples from SEC XBRL filings (P/E, EV/EBITDA, P/S, P/B,
P/FCF, ROIC, Piotroski F, Altman Z, accruals and more). Audit 12 added the **published composite and forensic scores**,
each exactly as its authors defined it, with nothing fitted:

| Score | Source | What it measures |
|---|---|---|
| Beneish M-score | Beneish (1999), eight indices | how much a firm's accounts look like those of known manipulators; flag above -1.78 |
| Montier C-score | Montier (2008), six signs | earnings-quality warning signs, 0 to 6 |
| Gross profitability | Novy-Marx (2013) | (revenue - cost of goods sold) / total assets |
| Magic formula | Greenblatt (2005) | earnings yield EBIT / EV together with return on capital |
| Quality (QMJ-style) | after Asness, Frazzini and Pedersen (2019) | profitability, one-year growth of profitability, safety |
| Value composite | classic value | mean percentile of earnings yield, book-to-price, EBITDA/EV, FCF yield, sales yield |
| Piotroski F | Piotroski (2000) | nine accounting signals |

**E30** asked the first question a factor researcher asks: *does each composite rank next month's returns in this
universe?* It was pre-registered before it was run: one test per composite, mean monthly Spearman rank correlation (IC)
with the next 20-session return, a one-sided month-block bootstrap, Holm correction over the six composites. This cell
repeats it from the staged table. SEC XBRL data begin in 2009, there are 164 month-ends (April 2009 to November 2022), and each composite's test uses the 137 to 151 months
with at least 100 stocks that have its score and a return.

In [ ]:
import importlib.util
E30_FILE = NB / "fundamentals" / "e30_stock_months.parquet"
if not E30_FILE.exists():
    raise FileNotFoundError(f"{E30_FILE} is missing: run scripts/colab/stage_e30_inputs.py on the PC "
                            "(it also copies the file to the Drive input folder)")
_spec = importlib.util.spec_from_file_location("nx_extended", REPO_DIR / "src" / "nexus_alpha" / "fundamentals" / "extended.py")
XF = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(XF)                       # numpy and pandas only: the published formulas, no fitted parameter

FM = pd.read_parquet(E30_FILE)
FM["date"] = pd.to_datetime(FM["date"])
if QUICK:
    FM = FM[FM.date.dt.year >= 2018]
print(f"{len(FM):,} stock-months, {FM.ticker.nunique()} stocks, {FM.date.min().date()} to {FM.date.max().date()}")
assert FM.date.max() < pd.Timestamp("2023-01-03"), "the sealed period must never be read"

FMX = pd.concat([FM.reset_index(drop=True), XF.metrics(FM.reset_index(drop=True))], axis=1)
cover = FMX[["beneish_m", "montier_c", "gross_profitability", "greenblatt_roc", "ebit_to_ev", "peg", "dso"]].notna().mean()
print("coverage of the new metrics (share of stock-months with a value):")
display(cover.round(2).to_frame("coverage").T)

**What the numbers look like on one stock.** Apple on the last session of 2021: each score from the as-known snapshot.
A score that needs a tag the company does not report (MSFT's depreciation, for instance) is missing, not guessed.

In [ ]:
ex = FMX[(FMX.ticker == "AAPL") & (FMX.date == FMX.date[FMX.ticker == "AAPL"].max())].iloc[0] if QUICK else \
     FMX[(FMX.ticker == "AAPL") & (FMX.date <= "2021-12-31")].sort_values("date").iloc[-1]
show = ["pe", "ev_ebitda", "ev_ebit", "peg", "gross_profitability", "greenblatt_roc", "roce", "cash_conversion", "dso",
        "dio", "quick_ratio", "beneish_m", "beneish_flag", "montier_c", "piotroski_f", "altman_z"]
display(pd.Series({k: (float(ex[k]) if k in ex and pd.notna(ex[k]) else np.nan) for k in show}, name=f"AAPL {ex.date.date()}").to_frame().T.round(3))

In [ ]:
PRIMARY = ["value_composite", "quality_composite", "gross_profitability_rank", "magic_formula", "piotroski_rank",
           "beneish_safety_rank"]
H_E30, BLOCK_E30, SEED_E30, MIN_STOCKS = 20, 6, 30, 100
N_BOOT_E30 = 500 if QUICK else 10_000

def spearman_(a, b):
    ok = a.notna() & b.notna()
    return float(a[ok].rank().corr(b[ok].rank())) if ok.sum() >= MIN_STOCKS else np.nan

monthly = []
for d, g_ in FMX.groupby("date"):
    g_ = g_.set_index("ticker")
    cs = XF.cross_section(g_, sic2=g_.sic2)                     # percentile composites within this month's universe
    row = {"date": d}
    for c in PRIMARY:
        row[c] = spearman_(cs[c], g_.R)                          # the month's rank IC with the next 20-session return
    monthly.append(row)
IC = pd.DataFrame(monthly).set_index("date").sort_index()

def block_p(v, block=BLOCK_E30, n_boot=N_BOOT_E30, seed=SEED_E30):
    # one-sided: the share of moving-block bootstrap means at or below zero (as evaluation.forward_criteria.block_p_value)
    v = np.asarray(pd.Series(v).dropna(), float); n = len(v); b = min(block, n); k = int(np.ceil(n / b))
    r = np.random.default_rng(seed)
    means = np.array([np.concatenate([v[s:s + b] for s in r.integers(0, n - b + 1, k)])[:n].mean() for _ in range(n_boot)])
    return float(np.mean(means <= 0))

def holm_(p):
    items = sorted(p.items(), key=lambda kv: kv[1]); out, run = {}, 0.0
    for i, (k, v) in enumerate(items):
        run = max(run, min(1.0, (len(items) - i) * v)); out[k] = run
    return out

def ci_(v, block=BLOCK_E30, n=2000, seed=SEED_E30):
    v = np.asarray(v.dropna(), float); b = block; k = int(np.ceil(len(v) / b)); r = np.random.default_rng(seed)
    m = [np.concatenate([v[s:s + b] for s in r.integers(0, len(v) - b + 1, k)])[:len(v)].mean() for _ in range(n)]
    return np.percentile(m, [2.5, 97.5])

P1 = {c: block_p(IC[c]) for c in PRIMARY}
HO = holm_(P1)
E30_TABLE = pd.DataFrame({c: {"months": int(IC[c].notna().sum()), "mean IC": IC[c].mean(),
                              "95% low": ci_(IC[c])[0], "95% high": ci_(IC[c])[1], "share of months > 0": (IC[c].dropna() > 0).mean(),
                              "p (one-sided)": P1[c], "p (Holm)": HO[c],
                              "replicates": bool(HO[c] < 0.05 and IC[c].mean() > 0)} for c in PRIMARY}).T
display(E30_TABLE.style.format({"mean IC": "{:+.4f}", "95% low": "{:+.4f}", "95% high": "{:+.4f}", "share of months > 0": "{:.2f}",
                                "p (one-sided)": "{:.3f}", "p (Holm)": "{:.3f}"}))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))
names = {"value_composite": "Value composite", "quality_composite": "Quality (QMJ-style)",
         "gross_profitability_rank": "Gross profitability", "magic_formula": "Magic formula",
         "piotroski_rank": "Piotroski F", "beneish_safety_rank": "Low Beneish M"}
y = np.arange(len(PRIMARY))[::-1]
est = E30_TABLE["mean IC"].astype(float).to_numpy(); lo = E30_TABLE["95% low"].astype(float).to_numpy(); hi = E30_TABLE["95% high"].astype(float).to_numpy()
ax[0].errorbar(est, y, xerr=[est - lo, hi - est], fmt="D", color="#1f77b4", ecolor="#9aa5b1", capsize=4)
ax[0].axvline(0, color="grey", ls=":")
ax[0].set_yticks(y); ax[0].set_yticklabels([names[c] for c in PRIMARY])
ax[0].set_xlabel("mean monthly rank IC with the next 20-session return"); ax[0].set_title("E30: no composite clears zero after Holm")
for c in ["gross_profitability_rank", "piotroski_rank", "value_composite"]:
    ax[1].plot(IC.index, IC[c].fillna(0).cumsum(), label=names[c])
ax[1].axhline(0, color="grey", ls=":"); ax[1].legend(); ax[1].set_title("cumulative monthly IC (a steady climb would be skill)")
plt.tight_layout(); plt.show()

In [ ]:
DOC30 = REPO_DIR / "reports" / "e30" / "e30_results.json"
if DOC30.exists() and not QUICK:
    doc = json.loads(DOC30.read_text(encoding="utf-8"))["primary"]
    diff = {c: abs(float(E30_TABLE.loc[c, "mean IC"]) - doc[c]["mean_ic"]) for c in PRIMARY}
    E30_PARITY = pd.DataFrame({"this run": E30_TABLE["mean IC"].astype(float), "documented": {c: doc[c]["mean_ic"] for c in PRIMARY},
                               "difference": diff})
    display(E30_PARITY.style.format("{:+.5f}"))
    print("max difference:", f"{max(diff.values()):.2e}", "(float32 staging explains anything below 1e-4)")
    assert max(diff.values()) < 5e-4, "the notebook no longer reproduces the documented E30 result"
else:
    print("documented result not compared (QUICK mode, or the repository clone has no reports/e30)")

**Reading it.** No composite has a mean IC whose 95% interval stays above zero after correcting for six tests: gross
profitability and Piotroski are the closest, value is about zero, and the growth and Montier parts lean the wrong way
(documented in `docs/E30_RESULTS.md`). Published anomalies weaken after publication and are weaker among large, liquid
stocks (McLean and Pontiff 2016 find returns 58% lower after publication), and this universe is today's large
survivors with price-only returns, which flatters junk and penalises high-yield stocks. The scores are therefore shown
on the dashboard (tab P) as a description of a company and a risk flag, not as a signal. A different horizon (one to
twelve months) would be a new hypothesis and needs its own pre-registration.

## 14. Comparison: every model side by side, and the reproduction check

The first chart compares the mean log-loss gain over the prior on test for every model in this notebook and every saved
documented run. The second compares chart encoders from E02 (a linear probe on each frozen backbone's embeddings,
weekly + daily + hourly), including the random-weights control and TriFrame's chart-fine-tuned ConvNeXt (which may be
scored only from 2017, so only validation and test are shown). The last table recomputes the documented E03 numbers
from the saved predictions and compares them with the committed `metrics.json`: they must agree.

In [ ]:
for k, f in (("E02 probe: random ResNet18 (control)", "e02__img_lr__resnet18_random_s0__WDH"),
             ("E02 probe: ResNet18", "e02__img_lr__resnet18__WDH"),
             ("E02 probe: ConvNeXt-Tiny", "e02__img_lr__convnext_tiny__WDH"),
             ("E02 probe: DINOv2 ViT-S", "e02__img_lr__vit_small_patch14_dinov2__WDH"),
             ("E02 probe: ConvNeXt-Tiny fine-tuned on charts", "e02__img_lr__ft_convnext_tiny__WDH")):
    if (NB / "predictions" / f"{f}.npy").exists():
        PRED[k] = saved(f)
gain = {}
for name, P in PRED.items():
    g = []
    for j in range(6):
        ok = YE[P_TEST, j] >= 0; ix = P_TEST[ok]
        if np.isnan(P[ix, j]).any():
            g = None; break
        g.append(log_loss(YE[ix, j], PRED["prior"][ix, j], labels=[0, 1, 2]) -
                 log_loss(YE[ix, j], np.clip(P[ix, j], 1e-7, 1), labels=[0, 1, 2]))
    if g is not None:
        gain[name] = np.mean(g)
gain = pd.Series(gain).sort_values()
gain.index = [LABEL.get(i, i) for i in gain.index]
fig, ax = plt.subplots(figsize=(9, 0.42 * len(gain) + 1))
ax.barh(gain.index, gain.to_numpy(), color=["#2ca02c" if v > 0 else "#d62728" for v in gain])
ax.axvline(0, color="k", lw=.8); ax.set_title("mean log-loss gain over the class prior on test (positive = better)")
plt.tight_layout(); plt.show()

repro = []
e03 = REPO_DIR / "experiments" / "20261106T000935_e03_fusion_8ddffb" / "metrics.json"
if e03.exists():
    M = json.loads(e03.read_text(encoding="utf-8"))
    for nm in ("variant_gated", "variant_late", "variant_attention", "a05_num", "a04_img_WDH", "selected_calibrated"):
        f = NB / "predictions" / f"e03__{nm}.npy"
        if not f.exists() or nm not in M:
            continue
        P = saved(f"e03__{nm}")
        for split, ix, key in (("valid", P_VAL, "valid"), ("test", P_TEST, "test_raw" if "test_raw" in M[nm] else "test")):
            if key not in M[nm]:
                continue
            ours = np.mean([log_loss(YE[ix[YE[ix, j] >= 0], j], np.clip(P[ix[YE[ix, j] >= 0], j], 1e-7, 1),
                                     labels=[0, 1, 2]) for j in range(6)])
            repro.append({"run": nm, "split": split, "recomputed here": ours,
                          "committed metrics.json": M[nm][key]["mean_log_loss"]})
    R = pd.DataFrame(repro)
    R["abs diff"] = (R["recomputed here"] - R["committed metrics.json"]).abs()
    REPRO = R                                                    # kept for the report (section 20)
    display(R.style.format({"recomputed here": "{:.5f}", "committed metrics.json": "{:.5f}", "abs diff": "{:.1e}"}))
    print("all within 2e-4 (float16 storage of the saved probabilities):", bool((R["abs diff"] < 2e-4).all()))

## 15. The risk engine: entry, stop, targets and size, with the reasons

Forecasting and trading are separate. A high P(UP) is never by itself a LONG. The engine applies its gates in order,
and the first failure decides (every reason is reported):

1. data sufficiency (exact levels need structured OHLCV; screenshots alone never give numbers);
2. a calibrated forecast for the horizon;
3. **validated and replicated direction skill** at the horizon: the up-vs-down AUROC interval must lie above 0.5 on
   the 2017-18 validation block (the frozen flag) **and**, since decision D-057 (audit 9), on 2020-2022 as well;
4. an edge |P(UP) - P(DOWN)| of at least 0.10;
5. the weekly regime must permit the side (rulebook E011-E018);
6. a daily setup on the same side supplies the structural invalidation and the first obstacle (E023/E027/E031...);
7. an hourly trigger on the as-of session (E121/E122), else WAIT;
8. stop = invalidation -/+ 0.25 ATR (X001, X011), on the losing side of the entry;
9. target 1 = the first opposing obstacle (X021); net reward-to-risk after costs must be at least 1.5 (E039, E149);
10. size = floor(capital x risk % / (risk per share + costs)), capped by the maximum position (X010);
11. a time exit after the horizon (X071); a maximum chase of 1 ATR beyond the reference (E111).

The engine never widens a stop to fit a size, never invents a level, and never reads prices from chart pixels. The
cell re-implements it and runs four cases:
* an approved long;
* the same setup at 1D, where v1.1's direction skill is not validated (gate 3);
* the same setup at 5D with v1.1's typical edge: on the latest logged session, the median |P(UP) - P(DOWN)| was 0.04
  at 5D, and 2.6% of stocks reached the 0.10 minimum;
* the same setup at 5D where the skill was validated on 2017-18 but did not replicate on 2020-22 (D-057);
* a short when shorting is disabled.

It then recomputes gate 3 with the frozen flags' own statistic (P(UP) - P(DOWN) on rows that moved, month-block
bootstrap) on validation, puts the frozen v1.1 flags next to it, and recomputes the same interval for v1.1 on test.
**No horizon is both validated and replicated**: the 2017-18 skill at 5D-1M did not hold on 2020-22, so the
dashboard's engine no longer trades direction on it (`src/nexus_alpha/trading/skill_evidence.py`). A horizon re-opens
only with forward evidence (120 matured sessions). The frozen models and their logged flags are unchanged.

In [ ]:
COST_BPS = {"base": 10, "stressed": 15, "severe": 20}            # round trip, plus 0.5% a year short borrow

def plan_trade(p_up, p_down, skill_ok, weekly_perm, setup, trigger, atr, close, sessions, capital=100_000,
               risk_pct=0.005, max_pos=0.10, allow_short=True, min_edge=0.10, min_rr=1.5, cost="base"):
    reasons = []
    edge = p_up - p_down; side = "LONG" if edge > 0 else "SHORT"
    if not skill_ok:
        reasons.append("no validated directional skill at this horizon")
    if abs(edge) < min_edge:
        reasons.append(f"edge {abs(edge):.3f} below {min_edge:.2f}")
    if side == "SHORT" and not allow_short:
        reasons.append("short selling disabled")
    ok_perm = {"LONG": {"LONG_PREFERRED", "EDGES_ONLY", "REDUCED_CONFIRMATION_REQUIRED"},
               "SHORT": {"SHORT_PREFERRED", "EDGES_ONLY", "REDUCED_CONFIRMATION_REQUIRED"}}[side]
    if weekly_perm not in ok_perm:
        reasons.append(f"weekly permission {weekly_perm} does not allow {side}")
    if setup is None or setup["direction"] != side:
        reasons.append(f"no daily {side.lower()} setup")
    if not trigger:
        reasons.append("no hourly trigger (WAIT)")
    if reasons:
        return {"decision": "NO_TRADE", "reasons": reasons}
    stop = setup["invalidation"] - 0.25 * atr if side == "LONG" else setup["invalidation"] + 0.25 * atr
    t1 = setup["first_obstacle"]
    if not ((stop < close < t1) if side == "LONG" else (t1 < close < stop)):
        return {"decision": "NO_TRADE", "reasons": ["stop/target geometry invalid"]}
    c_ps = close * COST_BPS[cost] / 1e4 + (close * 0.005 * sessions / 252 if side == "SHORT" else 0)
    risk_ps, reward_ps = abs(close - stop), abs(t1 - close)
    net_rr = (reward_ps - c_ps) / (risk_ps + c_ps)
    if net_rr < min_rr:
        return {"decision": "NO_TRADE", "reasons": [f"net reward-to-risk {net_rr:.2f} below {min_rr}"]}
    qty = min(math.floor(capital * risk_pct / (risk_ps + c_ps)), math.floor(capital * max_pos / close))
    if qty < 1:
        return {"decision": "NO_TRADE", "reasons": ["stop too wide for the risk budget (X010)"]}
    return {"decision": side, "entry (next open, if within chase)": close,
            "max chase": close + atr if side == "LONG" else close - atr, "stop": stop, "target 1": t1,
            "target 2": close + 2 * (t1 - close), "net R:R": net_rr, "quantity": qty, "position value": qty * close,
            "loss at stop": qty * (risk_ps + c_ps), "time exit (sessions)": sessions,
            "reasons": [f"setup {setup['family']} {setup['rule_ids']}: invalidation {setup['invalidation']:.2f}",
                        f"stop = invalidation - 0.25 ATR ({atr:.2f}) (X001, X011)", "T1 = first obstacle (X021)",
                        f"size: risk {risk_pct:.1%} of capital over (risk per share + costs) (X010)"]}

setup = {"family": "TREND_PULLBACK_LONG", "direction": "LONG", "rule_ids": ["E023", "E074", "E083"],
         "invalidation": 171.20, "first_obstacle": 189.50}
cases = {
    "approved long (skill validated, edge 0.16)": plan_trade(.42, .26, True, "LONG_PREFERRED", setup, True, 3.10, 176.40, 5),
    "same setup at 1D (direction skill not validated)": plan_trade(.42, .26, False, "LONG_PREFERRED", setup, True, 3.10, 176.40, 1),
    "same setup at 5D with v1.1's typical edge (0.04)": plan_trade(.31, .27, True, "LONG_PREFERRED", setup, True, 3.10, 176.40, 5),
    "same setup at 5D, validated 2017-18 but not replicated 2020-22 (D-057)": plan_trade(.42, .26, False, "LONG_PREFERRED", setup, True, 3.10, 176.40, 5),
    "short with shorting disabled": plan_trade(.25, .45, True, "SHORT_PREFERRED", None, True, 3.10, 176.40, 5, allow_short=False),
}
for k, v in cases.items():
    display(Markdown(f"**{k}**")); display(pd.Series({a: b for a, b in v.items()}, dtype=object).to_frame("value"))

def ud_auc_ci(P, ix_all, j, B=500, seed=7):
    # gate 3's statistic: AUROC of P(UP) - P(DOWN) on rows that moved; month-block bootstrap 95% interval
    ok = (YE[ix_all, j] >= 0) & (YE[ix_all, j] != 1); ix = ix_all[ok]; y = YE[ix, j] == 2
    s = P[ix, j, 2] - P[ix, j, 0]
    m = pd.DatetimeIndex(DATES[ix]).to_period("M").astype(str).to_numpy()
    groups = pd.Series(np.arange(len(ix))).groupby(m).indices; keys = list(groups)
    r_ = np.random.default_rng(seed); bs = []
    for _ in range(B):
        pick = np.concatenate([groups[keys[k]] for k in r_.integers(0, len(keys), len(keys))])
        bs.append(roc_auc_score(y[pick], s[pick]))
    lo_, hi_ = np.percentile(bs, [2.5, 97.5])
    return float(roc_auc_score(y, s)), float(lo_), float(hi_)

skill = {}
SK = "E03 gated (saved)" if "E03 gated (saved)" in PRED else "fusion"
for j, h in enumerate(H6):                                 # the gate-3 check, recomputed from validation predictions
    a_, lo_, hi_ = ud_auc_ci(PRED[SK], P_VAL, j, B=100 if QUICK else 500)
    skill[h] = {"valid up-vs-down AUROC (recomputed)": a_, "95% CI": f"[{lo_:.3f}, {hi_:.3f}]",
                "validated (recomputed)": lo_ > 0.5}
SKT = pd.DataFrame(skill).T
meta_f = WS / "nexus_artifacts" / "bundles" / "20261106T020411_e08_final_model_5c4327" / "bundle_meta.json"
if meta_f.exists():
    ds = json.loads(meta_f.read_text(encoding="utf-8"))["direction_skill"]
    ds = ds if isinstance(ds, dict) else __import__("ast").literal_eval(ds)
    SKT["frozen v1.1: AUROC [95% CI]"] = [f"{ds[h]['valid_up_vs_down_auroc']:.3f} [{ds[h]['ci95'][0]:.3f}, "
                                          f"{ds[h]['ci95'][1]:.3f}]" for h in SKT.index]
    SKT["frozen v1.1: validated (the flag the engine uses)"] = [ds[h]["validated"] for h in SKT.index]
test_auc = TABLE[(TABLE.model == "v1.1 (E08 frozen)") & (TABLE.split == "test")].set_index("horizon")["up-vs-down AUROC"]
SKT["v1.1 test up-vs-down AUROC"] = test_auc.reindex(SKT.index).round(3)
if "v1.1 (E08 frozen)" in PRED:                            # replication (D-057): the same statistic on 2020-22
    rep_ = {h: ud_auc_ci(PRED["v1.1 (E08 frozen)"], P_TEST, j, B=100 if QUICK else 500) for j, h in enumerate(H6)}
    SKT["v1.1 test AUROC [95% CI], gate statistic"] = [f"{rep_[h][0]:.3f} [{rep_[h][1]:.3f}, {rep_[h][2]:.3f}]"
                                                       for h in SKT.index]
    SKT["replicated on 2020-22"] = [rep_[h][1] > 0.5 for h in SKT.index]
    if "frozen v1.1: validated (the flag the engine uses)" in SKT:
        SKT["gate 3 open (validated and replicated)"] = (SKT["frozen v1.1: validated (the flag the engine uses)"].astype(bool)
                                                        & SKT["replicated on 2020-22"].astype(bool))
display(Markdown(f"**Gate 3, direction skill: recomputed on validation ({SK}, fitted 2003-2015, month-block "
                 "bootstrap), the frozen v1.1 flags, and the same interval for v1.1 on test (D-057)**"))
display(SKT)

## 16. Backtest (the documented run)

The event-driven backtest (`nexus_alpha.trading.backtest`) enters at the next open; stops that gap through fill at the
open; when a bar touches both the stop and the target, the stop is assumed first; costs are 10/15/20 bps round trip
plus 0.5% a year short borrow. Thresholds were chosen on validation and frozen; the test years are shown here. The
benchmark is equal-weight buy-and-hold of the same survivor universe, so it carries the same survivorship bias.

In [ ]:
bt = REPO_DIR / "reports" / "backtest" / "e04_backtest_v1.1.json"
eq = REPO_DIR / "reports" / "backtest" / "e04_daily_returns_v1.1.csv"
if bt.exists() and eq.exists():
    b = json.loads(bt.read_text(encoding="utf-8"))
    d = pd.read_csv(eq, index_col=0, parse_dates=True)
    curve = (1 + d).cumprod()
    fig, (a1, a2) = plt.subplots(2, 1, figsize=(11, 6), sharex=True, gridspec_kw={"height_ratios": [3, 1.2]})
    for c in curve.columns:
        bench = "benchmark" in c
        a1.plot(curve.index, curve[c], lw=2.4 if bench else 1.2, ls="--" if bench else "-", color="k" if bench else None, label=c)
        a2.plot(curve.index, curve[c] / curve[c].cummax() - 1, lw=1, color="k" if bench else None)
    a1.set_title("growth of 1, test 2020-2022, base costs"); a1.legend(fontsize=7, ncol=2); a2.set_ylabel("drawdown")
    plt.tight_layout(); plt.show()
    t = pd.DataFrame(b["test_final"]).T
    cols = [c for c in ["trades", "sharpe", "sharpe_ci95_month_blocks", "cagr", "max_drawdown", "win_rate",
                        "profit_factor", "expectancy_R", "psr_vs_0"] if c in t]
    BT_TEST, BT_JSON = t[cols], b                                # kept for the report (section 20)
    display(BT_TEST)
    print("benchmark (equal-weight survivors), test:", b["benchmark_ew_survivors"]["test"], "| PBO:", b.get("pbo"))
else:
    print("Backtest reports not found in the repository clone.")

## 16A. Entry, stop-loss and exit: one trading rule on every modality's forecasts (descriptive)

Section 16 backtests the documented strategies. Here a single, fixed rule turns each configuration's forecasts
(section 13A) and the frozen v1.1 model's into trades, so the results compare the inputs, not the rules:

* **Signal:** s = P(UP) - P(DOWN) at the horizon. On each decision date with at least 10 stocks, **long the top 20%**
  and **short the bottom 20%** (ranking needs no threshold, so nothing is tuned). The risk engine's own gate (|s| at
  least 0.10) is counted separately.
* **Entry:** the next session's open, the first executable price after the 09:00 New York decision time.
* **Primary bracket, "fixed 1.5/3 ATR":** stop-loss 1.5 ATR(14) from the entry, target 3 ATR (reward-to-risk 2:1),
  time exit at the close h sessions later. A bar that gaps through the stop fills at its open; a bar that touches both
  levels counts as a stop; a target fills at the target (conservative both ways). Textbook multiples, fixed before the
  first run.
* **Sensitivity bracket, "sqrt-time":** the same 2:1 bracket scaled with the square root of the holding period
  (stop 1.5 ATR x sqrt(h / 5), target twice that), so it equals the primary bracket at 5D, is wider at 20D (where the
  fixed stop catches routine noise) and tighter at 1D (where the fixed bracket almost never triggers). Reported, never
  chosen from: both brackets were declared before this version ran (audit 9, A-94).
* **Costs:** 10 bps round trip (base) and 20 bps (severe), plus 0.5% a year short borrow.

**Two null tests.** (1) **Random books:** 300 books with the same number of longs and shorts per date, picked at random.
(2) **Volatility-matched books:** 300 random books with the same number of longs and shorts in each (date, ATR-quintile)
cell as the model's book. Costs and R are scaled by each stock's ATR, so a model that merely prefers volatile stocks
would beat plain random books without any skill; the matched books remove that (A-91). p-values count
(r + 1) / (N + 1), so the smallest possible value is 1/301 = 0.003 (A-92). Beating the null books means "ranks better
than chance after the same costs", not "profitable": the "profitable" column requires the expectancy interval to lie
above zero.

**Reading R and bps.** R divides each trade's result by its own stop distance (1.5 ATR), so a quiet stock's trade
weighs more in R than a volatile stock's, and costs weigh more too. Average return in bps weighs every trade by capital.
The two can have opposite signs when quiet stocks lose and volatile stocks win (A-96). The table shows gross R (no
costs) and net R side by side, and the long leg against **every stock long at the same costs** (positive = the model's
longs beat buying everything).

**Sharpe ratio.** A daily mark-to-market book: each day's cohort gets 1/h of the capital (half long, half short),
positions are marked at every close, costs are charged on entry and borrow daily (A-95). Max drawdown on the same book.

The frozen v1.1 model is evaluated on **test only**: it was refitted on 2003-2018, so its 2017-18 rows are in-sample
(A-90). Survivor universe, as everywhere in this notebook.

In [ ]:
from concurrent.futures import ThreadPoolExecutor
TRD_H = ["1D", "5D", "20D"]
COST_BP = {"base": 10, "severe": 20}
BRACKETS = {"fixed 1.5/3 ATR": lambda h: (1.5, 3.0),
            "sqrt-time": lambda h: (1.5 * np.sqrt(HN[h] / 5), 3.0 * np.sqrt(HN[h] / 5))}
PRIMARY = "fixed 1.5/3 ATR"
N_NULL = 60 if QUICK else 300
EVP = np.concatenate([P_VAL, P_TEST]); FRI = EVI[EVP]
BARS16 = dict(globals().get("bars_cache") or {})
need_ = sorted(set(FR.ticker.iloc[FRI]) - set(BARS16))
need_ = [t for t in need_ if (BARS_DIR / f"{t}.parquet").exists()]
t0 = time.time()
with ThreadPoolExecutor(8) as ex_:
    for t_, d_ in zip(need_, ex_.map(read_daily, need_)):
        BARS16[t_] = d_
print(f"daily bars for {len(BARS16)} stocks ({len(need_)} read now, {time.time() - t0:.0f} s)")

HMAX = max(HN[h] for h in TRD_H)
NE = len(EVI)
PATH = {k: np.full((NE, HMAX), np.nan) for k in ("open", "high", "low", "close")}
PATH_DATE = np.full((NE, HMAX), np.datetime64("NaT"), dtype="datetime64[ns]")
sub = pd.DataFrame({"pos": EVP, "ticker": FR.ticker.to_numpy()[FRI], "asof_d": FR.asof_date.to_numpy()[FRI]})
for t_, g_ in sub.groupby("ticker"):
    d_ = BARS16.get(t_)
    if d_ is None:
        continue
    dates_ = d_["date"].to_numpy(dtype="datetime64[ns]")
    a_ = g_["asof_d"].to_numpy(dtype="datetime64[ns]")
    k_ = np.searchsorted(dates_, a_)
    hit = (k_ < len(dates_)) & (dates_[np.minimum(k_, len(dates_) - 1)] == a_)
    idx = k_[:, None] + 1 + np.arange(HMAX)[None, :]
    okk = hit[:, None] & (idx < len(dates_))
    idx = np.minimum(idx, len(dates_) - 1)
    for col in PATH:
        v = d_[col].to_numpy(float)[idx]
        PATH[col][g_.pos.to_numpy()] = np.where(okk, v, np.nan)
    PATH_DATE[g_.pos.to_numpy()] = np.where(okk, dates_[idx], np.datetime64("NaT"))
ENTRY = PATH["open"][:, 0]
ATR_E = (FR.d_atr_pct * FR.close).to_numpy(float)[EVI]
ATRP = FR.d_atr_pct.to_numpy(float)[EVI]
chk = np.isfinite(ENTRY[EVP]) & np.isfinite(FR.entry_open.to_numpy(float)[FRI])
TRD_CHECKS = {"valid + test rows": int(len(EVP)), "price paths found": float(np.isfinite(ENTRY[EVP]).mean()),
              "entry = stored next open (of rows with both)":
                  float(np.isclose(ENTRY[EVP][chk], FR.entry_open.to_numpy(float)[FRI][chk], rtol=1e-6).mean()),
              "first path day after the as-of session": bool((PATH_DATE[EVP, 0][np.isfinite(ENTRY[EVP])]
                                                              > FR.asof_date.to_numpy()[FRI][np.isfinite(ENTRY[EVP])]).all())}
print(TRD_CHECKS)
assert TRD_CHECKS["first path day after the as-of session"], "a trade would enter on or before its as-of session"

def bracket(side, h, k_stop, k_tgt):
    # outcome of a long (side +1) or short (-1) bracket trade on every row; independent of any model
    O, Hh, Lw, C = (PATH[k][:, :h] for k in ("open", "high", "low", "close"))
    risk = k_stop * ATR_E
    stop, tgt = ENTRY - side * risk, ENTRY + side * k_tgt * ATR_E
    hs = (Lw <= stop[:, None]) if side > 0 else (Hh >= stop[:, None])
    ht = (Hh >= tgt[:, None]) if side > 0 else (Lw <= tgt[:, None])
    BIG = h + 1
    fs = np.where(hs.any(1), hs.argmax(1), BIG); ft = np.where(ht.any(1), ht.argmax(1), BIG)
    nfin = np.isfinite(C).sum(1); last = np.maximum(nfin - 1, 0)
    r_ = np.arange(NE)
    stop_first = (fs < BIG) & (fs <= ft); tgt_first = (ft < BIG) & (ft < fs)
    o_at = O[r_, np.minimum(fs, h - 1)]
    gap = (fs > 0) & ((o_at < stop) if side > 0 else (o_at > stop))
    px = np.where(stop_first, np.where(gap, o_at, stop), np.where(tgt_first, tgt, C[r_, last]))
    day = np.where(stop_first, fs, np.where(tgt_first, ft, last)) + 1
    held = np.arange(h)[None, :] < day[:, None]
    fav = (Hh - ENTRY[:, None]) if side > 0 else (ENTRY[:, None] - Lw)
    adv = (ENTRY[:, None] - Lw) if side > 0 else (Hh - ENTRY[:, None])
    with np.errstate(invalid="ignore", divide="ignore"):
        out_ = {"R": side * (px - ENTRY) / risk, "ret": side * (px / ENTRY - 1), "px": px, "days": day, "risk": risk,
                "mfe": np.nanmax(np.where(held, fav, -np.inf), 1) / risk,
                "mae": np.nanmax(np.where(held, adv, -np.inf), 1) / risk,
                "exit": np.where(stop_first, "stop", np.where(tgt_first, "target", "time")),
                "ok": np.isfinite(ENTRY) & np.isfinite(ATR_E) & (ATR_E > 0) & (nfin >= h)}
    return out_

OUTC = {(b, h): {+1: bracket(+1, HN[h], *BRACKETS[b](h)), -1: bracket(-1, HN[h], *BRACKETS[b](h))}
        for b in BRACKETS for h in TRD_H}

def net(o, pos, side, cost):
    c = COST_BP[cost] / 1e4 + (0.005 * o["days"][pos] / 252 if side < 0 else 0)
    return o["ret"][pos] - c, o["R"][pos] - c * ENTRY[pos] / o["risk"][pos]

NETR = {(b, h, c): {s: net(OUTC[(b, h)][s], np.arange(NE), s, c)[1] for s in (+1, -1)}
        for b in BRACKETS for h in TRD_H for c in COST_BP}

def eligible(pos0, h):
    return pos0[OUTC[(PRIMARY, h)][+1]["ok"][pos0]]            # the ok rule does not depend on the bracket

def book_sides(score, E, q=0.2):
    df = pd.DataFrame({"pos": E, "d": DATES[E], "s": score[E]})
    df["n"] = df.groupby("d").s.transform("size"); df = df[df.n >= 10]
    df["r"] = df.groupby("d").s.rank(pct=True, method="first")
    return df.pos[df.r > 1 - q].to_numpy(), df.pos[df.r <= q].to_numpy()

def expectancy(L, S, b, h, cost="base"):
    nr = NETR[(b, h, cost)]
    return float((nr[+1][L].sum() + nr[-1][S].sum()) / max(len(L) + len(S), 1))

def mtm(L, S, b, h, cost="base"):
    # daily mark-to-market book: each cohort (decision date) gets 1/h of capital, half long and half short
    hh = HN[h]; parts_d, parts_r = [], []
    for side, pos in ((+1, L), (-1, S)):
        if not len(pos):
            continue
        o = OUTC[(b, h)][side]
        C = PATH["close"][pos, :hh]; D = PATH_DATE[pos, :hh]; e = o["days"][pos]; px = o["px"][pos]
        prev = np.concatenate([ENTRY[pos][:, None], C[:, :-1]], 1)
        k = np.arange(hh)[None, :]
        cur = np.where(k == (e - 1)[:, None], px[:, None], C)
        r = side * (cur / prev - 1)
        cohort = pd.Series(DATES[pos]); n = cohort.map(cohort.value_counts()).to_numpy()
        w = 1.0 / (hh * 2 * n)
        c = np.zeros_like(r); c[:, 0] = COST_BP[cost] / 1e4
        if side < 0:
            c = c + 0.005 / 252
        act = (k < e[:, None]) & np.isfinite(r)
        parts_d.append(D[act]); parts_r.append((w[:, None] * (r - c))[act])
    if not parts_d:
        return pd.Series(dtype=float)
    return pd.Series(np.concatenate(parts_r)).groupby(np.concatenate(parts_d)).sum().sort_index()

def book_stats(daily):
    if len(daily) < 20 or daily.std() == 0:
        return np.nan, np.nan, np.nan
    eq = (1 + daily).cumprod()
    return (float(daily.mean() / daily.std() * np.sqrt(252)), float(daily.mean() * 252),
            float((eq / eq.cummax() - 1).min()))

def trade_stats(L, S, b, h, cost="base"):
    oL, oS = OUTC[(b, h)][+1], OUTC[(b, h)][-1]
    rl, Rl = net(oL, L, +1, cost); rs, Rs = net(oS, S, -1, cost)
    R = np.concatenate([Rl, Rs]); r = np.concatenate([rl, rs]); d = np.concatenate([DATES[L], DATES[S]])
    if len(R) == 0:
        return {"trades": 0}
    Rg = np.concatenate([oL["R"][L], oS["R"][S]])
    # the round-trip cost per trade at which the book would break even (gross return after short borrow, in bps)
    be = np.concatenate([oL["ret"][L], oS["ret"][S] - 0.005 * oS["days"][S] / 252]).mean() * 1e4
    ex = np.concatenate([oL["exit"][L], oS["exit"][S]])
    est, ci, _ = month_block_boot(R, d)
    pf = r[r > 0].sum() / max(-r[r < 0].sum(), 1e-12)
    sh, ar, mdd = book_stats(mtm(L, S, b, h, cost))
    return {"trades": len(R), "win rate": float((r > 0).mean()), "target hit": float((ex == "target").mean()),
            "stopped out": float((ex == "stop").mean()), "time exit": float((ex == "time").mean()),
            "expectancy R (gross)": float(np.mean(Rg)), "expectancy R": est, "R low": float(ci[0]), "R high": float(ci[1]),
            "profitable (R interval above 0)": bool(ci[0] > 0),
            "avg net return (bps)": float(r.mean() * 1e4), "break-even round trip (bps)": float(be),
            "profit factor": float(pf),
            "MFE (R)": float(np.nanmean(np.concatenate([oL["mfe"][L], oS["mfe"][S]]))),
            "MAE (R)": float(np.nanmean(np.concatenate([oL["mae"][L], oS["mae"][S]]))),
            "avg sessions held": float(np.concatenate([oL["days"][L], oS["days"][S]]).mean()),
            "long leg R": float(Rl.mean()) if len(Rl) else np.nan, "short leg R": float(Rs.mean()) if len(Rs) else np.nan,
            "Sharpe (daily MTM)": sh, "annual return (daily MTM)": ar, "max drawdown (daily MTM)": mdd}

def pval(null, x):
    return float((1 + (np.asarray(null) >= x).sum()) / (1 + len(null)))

def random_null(E, b, h, n, seed=SEED):
    rng_ = np.random.default_rng(seed)
    return np.array([expectancy(*book_sides(rng_.random(NE), E), b, h) for _ in range(n)])

def matched_null(L, S, E, b, h, n, seed=SEED):
    # random books with the model's number of longs and shorts in every (date, ATR-quintile-within-date) cell (A-91)
    df = pd.DataFrame({"d": DATES[E], "a": ATRP[E]})
    qn = np.minimum((df.groupby("d").a.rank(method="first", pct=True).to_numpy() * 5).astype(int), 4)
    cell = pd.factorize(df.d)[0] * 5 + qn
    at = np.full(NE, -1); at[E] = np.arange(len(E))
    ncell = int(cell.max()) + 1
    nL = np.bincount(cell[at[L]], minlength=ncell); nS = np.bincount(cell[at[S]], minlength=ncell)
    cs = np.sort(cell); rank = np.arange(len(E)) - np.searchsorted(cs, cs, side="left")
    isL = rank < nL[cs]; isS = (rank >= nL[cs]) & (rank < nL[cs] + nS[cs])
    rng_ = np.random.default_rng(seed); out_ = np.empty(n)
    for i in range(n):
        order = np.lexsort((rng_.random(len(E)), cell))
        out_[i] = expectancy(E[order[isL]], E[order[isS]], b, h)
    return out_

TRD_MODELS = {c: PV[c] for c in SC_ORDER if c != "prior"}
V11 = "v1.1 (frozen, production)"
if "v1.1 (E08 frozen)" in PRED:
    TRD_MODELS[V11] = PRED["v1.1 (E08 frozen)"]
TRD_ROWS, NULLS = [], {}
t0 = time.time()
for b in BRACKETS:
    for split, pos0 in (("test", P_TEST), ("valid", P_VAL)):
        for h in TRD_H:
            j = H6.index(h); ELIG = eligible(pos0, h)
            NULLS[(b, split, h)] = random_null(ELIG, b, h, N_NULL)
            all_long = float(NETR[(b, h, "base")][+1][ELIG].mean())
            L_, S_ = book_sides(np.random.default_rng(1).random(NE), ELIG)
            TRD_ROWS.append({"bracket": b, "split": split, "horizon": h, "model": "random book (= prior)",
                             **trade_stats(L_, S_, b, h), "expectancy R (severe costs)": expectancy(L_, S_, b, h, "severe"),
                             "every stock long (R, net)": all_long, "edge-gate trades (|s| >= 0.10)": 0})
            for name, P in TRD_MODELS.items():
                if name == V11 and split == "valid":                     # in-sample (refit 2003-2018): not shown
                    continue
                s = P[:, j, 2] - P[:, j, 0]
                L_, S_ = book_sides(s, ELIG)
                st_ = trade_stats(L_, S_, b, h)
                mn = matched_null(L_, S_, ELIG, b, h, N_NULL)
                TRD_ROWS.append({"bracket": b, "split": split, "horizon": h, "model": name, **st_,
                                 "expectancy R (severe costs)": expectancy(L_, S_, b, h, "severe"),
                                 "p vs random books": pval(NULLS[(b, split, h)], st_["expectancy R"]),
                                 "p vs vol-matched books": pval(mn, st_["expectancy R"]),
                                 "every stock long (R, net)": all_long,
                                 "long leg minus every stock long (R)": st_["long leg R"] - all_long,
                                 "edge-gate trades (|s| >= 0.10)": int((np.abs(s[ELIG]) >= 0.10).sum())})
        print(f"{b}, {split}: {time.time() - t0:.0f} s")
TRD = pd.DataFrame(TRD_ROWS).set_index(["bracket", "split", "horizon", "model"])
TFMT = {c: "{:.3f}" for c in TRD.columns if TRD[c].dtype != bool} | {
    "trades": "{:,.0f}", "edge-gate trades (|s| >= 0.10)": "{:,.0f}", "avg net return (bps)": "{:+.1f}",
    "break-even round trip (bps)": "{:+.1f}",
    "expectancy R": "{:+.3f}", "expectancy R (gross)": "{:+.3f}", "R low": "{:+.3f}", "R high": "{:+.3f}",
    "long leg minus every stock long (R)": "{:+.3f}", "Sharpe (daily MTM)": "{:+.2f}", "annual return (daily MTM)": "{:+.2%}",
    "max drawdown (daily MTM)": "{:.1%}"}
TRD_TEST, TRD_VALID = TRD.loc[(PRIMARY, "test")], TRD.loc[(PRIMARY, "valid")]
TRD_SCALED = TRD.loc["sqrt-time"]
COMPACT16 = ["trades", "win rate", "stopped out", "expectancy R (gross)", "expectancy R", "R low", "R high",
             "profitable (R interval above 0)", "p vs random books", "p vs vol-matched books",
             "long leg minus every stock long (R)", "break-even round trip (bps)", "Sharpe (daily MTM)",
             "max drawdown (daily MTM)"]
display(Markdown("**Test 2020-22, primary bracket (fixed 1.5/3 ATR): every model's forecasts** (base costs)"))
display(TRD_TEST.style.format(TFMT, na_rep="-").background_gradient(cmap="RdYlGn", subset=["expectancy R"], vmin=-.1, vmax=.1))
display(Markdown("**Validation 2017-18, primary bracket** (v1.1 omitted: in-sample)"))
display(TRD_VALID[[c for c in COMPACT16 if c in TRD_VALID]].style.format(TFMT, na_rep="-"))
display(Markdown("**Sensitivity: the horizon-scaled bracket (sqrt-time), test and validation**"))
display(TRD_SCALED[[c for c in COMPACT16 if c in TRD_SCALED]].style.format(TFMT, na_rep="-"))
print(f"trade simulation: {time.time() - t0:.0f} s")

In [ ]:
# where the edge would be: expectancy with intervals against the random books, the daily book, the stop diagnostics
fig, axs = plt.subplots(1, len(TRD_H), figsize=(6 * len(TRD_H), 4.4), sharey=True)
for ax, h in zip(axs, TRD_H):
    t_ = TRD_TEST.loc[h].drop(index="random book (= prior)")
    yy = np.arange(len(t_))
    lo, hi = np.percentile(NULLS[(PRIMARY, "test", h)], [2.5, 97.5])
    ax.axvspan(lo, hi, color="#9aa5b1", alpha=.3, label="95% of random books")
    ax.errorbar(t_["expectancy R"], yy, xerr=[t_["expectancy R"] - t_["R low"], t_["R high"] - t_["expectancy R"]],
                fmt="o", color="#d62728", capsize=3)
    ax.axvline(0, color="k", lw=.8); ax.set_yticks(yy, t_.index); ax.set_title(f"{h}: expectancy per trade (R, net), test")
    ax.legend(fontsize=7, loc="lower right")
plt.tight_layout(); plt.show()

h = "5D"; E5 = eligible(P_TEST, h)
fig, axs = plt.subplots(1, 2, figsize=(14, 4))
for name, col in ((ALLK, "#d62728"), ("vision only", "#1f77b4"), ("NLP only", "#9467bd"), ("indicators only", "#2ca02c"),
                  (V11, "#ff7f0e")):
    if name not in TRD_MODELS:
        continue
    s = TRD_MODELS[name][:, H6.index(h), 2] - TRD_MODELS[name][:, H6.index(h), 0]
    daily = mtm(*book_sides(s, E5), PRIMARY, h)
    axs[0].plot(daily.index, (1 + daily).cumprod(), label=name, color=col, lw=1.3)
axs[0].axhline(1, color="k", lw=.6)
axs[0].set_title(f"daily mark-to-market long-short book, {h}, test (growth of 1, base costs)")
axs[0].legend(fontsize=8)
o = OUTC[(PRIMARY, h)][+1]; okp = E5
win = o["R"][okp] > 0
axs[1].hist(o["mae"][okp][win], bins=np.linspace(0, 3, 31), alpha=.6, label="winning longs", color="#2ca02c", density=True)
axs[1].hist(o["mae"][okp][~win], bins=np.linspace(0, 3, 31), alpha=.6, label="losing longs", color="#d62728", density=True)
axs[1].axvline(1.0, color="k", ls=":", lw=1)
axs[1].set_title(f"maximum adverse excursion before the exit (in R = 1.5 ATR), every stock, {h}, test")
axs[1].set_xlabel("MAE (R); 1.0 = the stop"); axs[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

stop_rows = {}
for b in BRACKETS:
    for h in TRD_H:
        o = OUTC[(b, h)][+1]; E_ = eligible(P_TEST, h)
        stop_rows[(b, h)] = {"long: stopped out": float((o["exit"][E_] == "stop").mean()),
                             "long: target hit": float((o["exit"][E_] == "target").mean()),
                             "long: winners that first went 0.75 R against": float((o["mae"][E_] > .75)[o["R"][E_] > 0].mean()),
                             "every stock long: mean R gross (market drift)": float(np.nanmean(o["R"][E_])),
                             "every stock long: mean R net (10 bps)": float(np.nanmean(NETR[(b, h, "base")][+1][E_])),
                             "cost of a round trip in R (mean)": float(np.nanmean(o["R"][E_] - NETR[(b, h, "base")][+1][E_]))}
TRD_STOP = pd.DataFrame(stop_rows)
display(Markdown("**Stop-loss diagnostics, every test stock-date, long bracket** (no model: the market alone; gross and "
                 "net of the same costs as the model rows)"))
display(TRD_STOP.style.format("{:.3f}"))

## 17. The knowledge base (RAG)

The books on technical analysis and the two rulebooks (150 entry rules, 120 exit rules) are split into chunks with
their page or rule id. Three retrievers: **BM25** (keyword, k1 = 1.5, b = 0.75, implemented below), **dense** (BGE-small
sentence embeddings, cosine similarity) and **hybrid** (reciprocal-rank fusion of the two). RAG explains; it never
changes a probability (tested in the repository: identical outputs with RAG on or off). Snippets are cut at 160
characters because the books are copyrighted.

In [ ]:
import re
from collections import Counter, defaultdict
CH = pd.read_parquet(NB / "rag" / "chunks.parquet")
DV = np.load(NB / "rag" / "dense_bge_small.npy")
TOKEN = re.compile(r"[a-z0-9]+")
STOP = set("a an the of and or to in on for with by is are was were be been it its this that these those as at from into "
           "than then so such can may not no but if when which who what how why where there their they them".split())
tok = lambda t: [w for w in TOKEN.findall((t or "").lower()) if w not in STOP and len(w) > 1]

class BM25:
    def __init__(self, docs, k1=1.5, b=0.75):
        self.k1, self.b = k1, b
        self.tf = [Counter(tok(d)) for d in docs]
        self.len = np.array([sum(t.values()) for t in self.tf], float); self.avg = self.len.mean()
        df = Counter(w for t in self.tf for w in t)
        n = len(docs); self.idf = {w: math.log(1 + (n - c + 0.5) / (c + 0.5)) for w, c in df.items()}
        self.post = defaultdict(list)
        for i, t in enumerate(self.tf):
            for w in t:
                self.post[w].append(i)
    def scores(self, q):
        s = np.zeros(len(self.tf))
        for w in set(tok(q)):
            for i in self.post.get(w, []):
                f = self.tf[i][w]
                s[i] += self.idf[w] * f * (self.k1 + 1) / (f + self.k1 * (1 - self.b + self.b * self.len[i] / self.avg))
        return s

bm = BM25(CH.text.tolist())
dense_ok = False
try:
    from transformers import AutoModel, AutoTokenizer
    TK = AutoTokenizer.from_pretrained("BAAI/bge-small-en-v1.5"); EM = AutoModel.from_pretrained("BAAI/bge-small-en-v1.5").eval()
    def qvec(q):
        with torch.no_grad():
            e = TK(["Represent this sentence for searching relevant passages: " + q], return_tensors="pt", truncation=True)
            return F.normalize(EM(**e).last_hidden_state[:, 0], dim=-1).numpy()[0]
    dense_ok = True
except Exception as e:
    print("dense retriever unavailable:", type(e).__name__)

def search(q, k=5, method="hybrid"):
    b_ = bm.scores(q)
    if method == "bm25" or not dense_ok:
        sc = b_
    else:
        d_ = DV @ qvec(q)
        if method == "dense":
            sc = d_
        else:
            sc = np.zeros(len(CH))
            for ranks in (np.argsort(-b_), np.argsort(-d_)):
                sc[ranks] += 1 / (60 + np.arange(1, len(ranks) + 1))
    top = np.argsort(-sc)[:k]
    r = CH.iloc[top][["doc_id", "page", "rule_id", "section"]].copy()
    r["score"] = sc[top]; r["snippet"] = CH.text.iloc[top].str.slice(0, 160).str.replace("\n", " ") + "..."
    return r

for q in ["bullish engulfing at support: where to place the stop", "ATR based trailing stop for a breakout",
          "weekly trend permission before a daily long setup"]:
    display(Markdown(f"**Query:** {q}")); display(search(q, 4))
ev = REPO_DIR / "reports" / "rag" / "rag_eval.json"
if ev.exists():
    E = json.loads(ev.read_text(encoding="utf-8"))
    display(Markdown("**Documented retrieval evaluation** (35 book questions with gold pages; rule lookups)"))
    display(pd.DataFrame(E["book"]).T[["recall@1", "recall@5", "recall@10", "mrr", "ndcg@10"]].style.format("{:.3f}"))

## 18. The forward test: verified from the public repository

Every night the frozen models forecast all ~500 stocks for the session that just closed, before the decision time
(09:00 New York on the next session). Each forecast file is SHA-256 hashed into a chain (each row includes the hash of
the previous row), and the chain is pushed to GitHub, which fixes its time publicly. Nothing can be edited later
without breaking the chain. The cell recomputes every chain from the clone and lists what has been logged.

**Two additions since v4.** (1) `v1.2` is the v1.1 recipe, unchanged, refit on 2003 to 2021-11-30 and calibrated on
2022 (E31, D-062): a single forward test of whether refitting on recent data helps. (2) `v1.1_news` logs the frozen v1.1
*with* the archived live headlines, for the stocks that have one (E29b, D-058): a first live test of the text branch.
Both are judged only after 120 sessions, and both have their own hash chains, verified below as soon as they exist.

In [ ]:
FIELDS = ["run_utc", "asof_session", "file", "file_sha256", "n_rows", "n_failed", "late_rows"]
def verify_chain(version):
    # row_hash = sha256(previous row_hash + "|" + the row's seven fields joined by "|"); the first row links to GENESIS
    ch = pd.read_csv(REPO_DIR / "shadow" / version / "chain.csv", dtype=str, keep_default_na=False)
    problems, prev = [], "GENESIS"
    for _, r in ch.iterrows():
        f = REPO_DIR / "shadow" / version / r.file           # r.file already starts with "forecasts/"
        if not f.exists():
            problems.append(f"{r.file}: missing")
        elif sha256(f) != r.file_sha256:
            problems.append(f"{r.file}: file hash differs")
        expect = hashlib.sha256((prev + "|" + "|".join(str(r[c]) for c in FIELDS)).encode()).hexdigest()
        if r.prev_hash != prev or r.row_hash != expect:
            problems.append(f"{r.file}: chain link differs")
        prev = r.row_hash
    return ch, problems

if (REPO_DIR / "shadow").exists():
    out = []
    for v in ["v1.1", "v2", "market_e19", "v3_fund", "v3_alpha", "v1.1_news", "v1.2"]:
        if not (REPO_DIR / "shadow" / v / "chain.csv").exists():
            continue
        ch, probs = verify_chain(v)
        out.append({"model": v, "sessions logged": ch.asof_session.nunique(), "first": ch.asof_session.min(),
                    "latest": ch.asof_session.max(), "late rows": int(ch.late_rows.astype(int).sum()),
                    "chain": "intact" if not probs else "; ".join(probs[:2])})
    FWD = pd.DataFrame(out)                                      # kept for the report (section 20)
    display(FWD)
    print("The promotion rule needs about 120 sessions before any claim (docs/FORWARD_SHADOW_TEST.md).")

## 19. Conclusions

**What the evidence supports**
* The pipeline is point in time, purged and reproducible: labels, splits and indicators are re-derived above and match.
* Logistic regression reaches its certified global minimum; LightGBM and the networks stop where held-out loss stops
  improving; the seeds land in equally good minima.
* Volatility regime (sideways vs moves) is predictable: the sideways AUROC is clearly above 0.5 at every horizon
  (section 12).
* The frozen v1.1 probabilities beat the class prior on test at every horizon, significantly at the short horizons
  after Holm (section 13). Their calibration is looser on test than on validation (ECE up to about 0.04 at 10D-1M):
  calibration fitted on 2019 does not fully transfer (audit finding A-85).
* Cross-sectional ranking has a small positive rank IC, the part kept for the forward test.

**What it does not support**
* Single-stock direction at 1-20 sessions. On validation the up-vs-down AUROC was 0.51-0.54, with intervals just
  above 0.5 from 5D; on the 2020-2022 test years it was 0.49-0.51 (sections 12 and 15). The small validation skill did
  not replicate, at any horizon where it had been validated (section 15). Since decision D-057 the dashboard's risk
  engine therefore requires both blocks and plans no directional trade; a horizon re-opens only with forward evidence.
* Charts, news and filings add little beyond the indicators (section 14; E03's H1-H3 are not significant after Holm).
* Transformer language does not change that (sections 8A-8C, E28). FinBERT news, ChronoBERT filing vectors and the SEC
  price-impact expert beat a within-date placebo on validation (+0.00023 [+0.00003, +0.00040]) but not the current six
  blocks (-0.00029 [-0.00082, +0.00025]): NOT KEPT. The language is stock-specific but redundant with the charts and
  indicators. The SEC expert reproduces the SEC study inside NEXUS: filings predict whether a stock moves (AUC 0.656)
  but not which way (0.513).
* Fine-tuning the vision network on charts does not beat the prior (section 10), as TriFrame found.
* Vision, NLP and indicators separately and together (section 13A). In the A100 run of 2026-11-13 (report committed in
  `reports/verification/`) the indicators carried most of the gain (+0.010 over the prior on test), the charts alone
  +0.004 and text alone +0.001 (not significant); the charts added nothing once indicators and text were present, and
  text added less than the seed-to-seed spread. Direction stayed at chance for every input.
* Entry, stop-loss and exit (section 16A). In the same run no input's long-short book had an expectancy interval above
  zero after costs at any horizon. The indicator-based books lost less than random books: the ranking signal is real
  but smaller than a 10 bps round trip. The volatility-matched null, the daily mark-to-market book and the sqrt-time
  bracket added in audits 9-11 show whether that survives once volatility is controlled. Both sections are
  descriptive: nothing is selected from them.
* No backtest edge over equal-weight buy-and-hold after costs (section 16).
* Published fundamental scores (section 13B, E30). Value, QMJ-style quality, gross profitability, the magic formula,
  Piotroski and Beneish were tested once, point in time, on 164 month-ends (137-151 months per composite, 2009-2022): none ranks the
  next 20-session return after Holm correction (the closest, gross profitability, has a mean IC of +0.018 with a
  95% interval that includes zero). They describe a company; they do not predict this universe's month.

**What comes next**
* The forward test runs nightly; about 120 sessions are needed before any promotion (section 18). It now also logs
  the E29b news-on forecasts (does live news help?) and v1.2 (does refitting on recent data help?).
* The honest ceiling: the edge that exists is about the size of trading costs. Making it profitable needs evidence the
  forward test has not yet produced, survivorship-free data, dividend-adjusted returns and a realistic cost model.
* The sealed 2023+ period is opened once, by the owner, after the final freeze.
* `docs/LIMITATIONS.md` lists every known limit (survivorship, vendor differences, reused test years).

In [ ]:
print(f"notebook finished in {(time.time() - T_START) / 60:.1f} minutes on {DEVICE}"
      + (" (QUICK mode: small samples)" if QUICK else ""))

## 20. Full report: every key number of this run in one file

The last cell collects the tables of every section (data, splits, labels, E28, the language branch, the SEC expert,
optimisation, fine-tuning, calibration, every metric, the statistics, the modality scorecard with seeds, coverage and
year-by-year stability, the trading evaluation with both null tests and both brackets, the backtest, the knowledge base
the fundamentals replication and the forward test) and writes them to Google Drive as one Markdown file and one JSON file:
`antigravity_workspace/nexus_artifacts/notebook_reports/NOTEBOOK_REPORT_<UTC time>.md` (and `.json`).

Open the Markdown file on Drive, or share it, to review the whole run without scrolling through the notebook. A section
that did not run is listed as missing rather than failing the cell. Notes on how to read the tables come first.

In [ ]:
import datetime as _dt, platform
REPORT_DIR = Path(os.environ.get("NEXUS_REPORT_DIR", str(WS / "nexus_artifacts" / "notebook_reports")))
REPORT_DIR.mkdir(parents=True, exist_ok=True)
STAMP = _dt.datetime.now(_dt.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
G_ = globals()

def _git(*a):
    try:
        return subprocess.run(["git", "-C", str(REPO_DIR), *a], capture_output=True, text=True, timeout=20).stdout.strip()
    except Exception:
        return "unknown"

def _cell(v):
    if isinstance(v, (bool, np.bool_)):
        return "yes" if v else "no"
    if isinstance(v, (float, np.floating)):
        return "-" if not np.isfinite(v) else (f"{v:.5f}" if abs(v) < 100 else f"{v:,.1f}")
    if isinstance(v, (list, tuple, np.ndarray)):
        return "[" + ", ".join(_cell(x) for x in list(v)[:8]) + "]"
    return str(v).replace("|", "/").replace("\n", " ")

def md_table(obj, max_rows=400):
    df = obj if isinstance(obj, pd.DataFrame) else (obj.to_frame() if isinstance(obj, pd.Series) else pd.Series(obj).to_frame("value"))
    df = df.copy()
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = [" / ".join(map(str, c)) for c in df.columns]
    df = df.reset_index()
    df.columns = [" / ".join(map(str, c)) if isinstance(c, tuple) else str(c) for c in df.columns]
    # A-109: a "|" inside a column name (e.g. "|s| >= 0.10") would split the header and shift every column
    lines = ["| " + " | ".join(c.replace("|", "\\|") for c in df.columns) + " |", "|" + "---|" * len(df.columns)]
    for _, r in df.head(max_rows).iterrows():
        lines.append("| " + " | ".join(_cell(v) for v in r.to_numpy()) + " |")
    if len(df) > max_rows:
        lines.append(f"\n({len(df) - max_rows} more rows in the JSON file)")
    return "\n".join(lines)

def jsonable(obj):
    if isinstance(obj, pd.DataFrame):
        d = obj.copy()
        if isinstance(d.columns, pd.MultiIndex):
            d.columns = [" / ".join(map(str, c)) for c in d.columns]
        d = d.reset_index()
        d.columns = [" / ".join(map(str, c)) if isinstance(c, tuple) else str(c) for c in d.columns]
        return json.loads(d.to_json(orient="records", default_handler=str))
    if isinstance(obj, pd.Series):
        return json.loads(obj.to_json(default_handler=str))
    return json.loads(json.dumps(obj, default=lambda o: o.item() if hasattr(o, "item") else str(o)))

def g(name, f=None):
    if name not in G_:
        return None
    try:
        return f(G_[name]) if f else G_[name]
    except Exception as e:
        return {"error": f"{type(e).__name__}: {e}"}

NOTES = ("* **In-sample rows.** v1.1 was refitted on 2003-2018: its 2017-18 (validation) numbers in section 12 are "
         "in-sample and marked so; section 16A shows v1.1 on test only.\n"
         "* **Test years.** 2020-22 is a reused development test (TriFrame read it; every model since v1 was scored on "
         "it). It is a diagnostic, not proof; only the forward test and the sealed 2023+ evaluation are clean.\n"
         "* **p-values.** Section 13A: one-sided month-block bootstrap (the share of resamples at or below zero), Holm-"
         "adjusted over the eight comparisons of each row set. Section 16A: (r + 1) / (N + 1) against 300 null books "
         "(smallest possible 0.003); beating the null books means ranking better than chance after the same costs, "
         "not profitability (see 'profitable').\n"
         "* **Seeds.** Every fusion number moves by about 0.001 between seeds; differences smaller than the seed SD in "
         "13A are not distinguishable.\n"
         "* **R versus bps.** R divides by each trade's stop distance, so quiet stocks weigh more; bps weigh every trade "
         "by capital. Opposite signs mean quiet stocks lost while volatile stocks won.\n"
         "* **Direction skill (gate 3).** No horizon of v1.1 or v2 is both validated (2017-18) and replicated "
         "(2020-22); the dashboard's engine therefore trades no direction (D-057). The frozen models are unchanged.\n"
         "* **Forward test.** Late rows are flagged and never scored; v3_fund's 9 late rows are the 1-stock file of "
         "session 2026-11-05 (incident D-045).")

SECTIONS = [
    ("Run", {"started": _dt.datetime.fromtimestamp(T_START, _dt.timezone.utc).isoformat(timespec="seconds"),
             "minutes": round((time.time() - T_START) / 60, 1), "QUICK mode": QUICK, "device": DEVICE,
             "gpu": torch.cuda.get_device_name(0) if DEVICE == "cuda" else "none", "python": platform.python_version(),
             "torch": torch.__version__, "lightgbm": lgb.__version__, "pandas": pd.__version__, "numpy": np.__version__,
             "repository commit": _git("rev-parse", "--short", "HEAD"), "commit date": _git("log", "-1", "--format=%cI"),
             "notebook build": "v5, audit 12 (2026-11-14)", "seeds (sections 8, 8C)": SEEDS,
             "seeds (section 13A)": g("SEEDS13")}),
    ("How to read this report", NOTES),
    ("2. Data", {"rows": len(FR), "stocks": int(FR.ticker.nunique()), "first": str(FR.asof_date.min().date()),
                 "last": str(FR.asof_date.max().date()), **FR.block.fillna("purged").value_counts().to_dict()}),
    ("3. Labels: recomputed vs stored", g("mism", pd.DataFrame)),
    ("4. Splits: purging and embargo between blocks", g("rep", pd.DataFrame)),
    ("6. Logistic regression: global-minimum certificate", g("CERT")),
    ("7. LightGBM: best boosting round per horizon (early stop on the inner block)", g("BEST_IT")),
    ("8. Fusion network: epochs and best inner loss per seed",
     g("HIST", lambda H: {s: {"epochs": len(h), "best inner CE": float(h["inner CE (early stop)"].min())} for s, h in H.items()})),
    ("8A. ChronoBERT re-encoding parity (centred cosine)",
     g("COS", lambda c: {"filings": int(len(c)), "median": float(np.median(c)), "1st percentile": float(np.percentile(c, 1)),
                         "minimum": float(np.min(c))})),
    ("8B. SEC price-impact expert", g("SI_AUC", pd.DataFrame)),
    ("8C. E28 vision + NLP keep test (this run)", g("G28")),
    ("8C. E28 documented run: configurations", g("T28")),
    ("8C. E28 documented run: intervals", g("E28_D")),
    ("8C. E28 documented run: rank IC", g("RIC")),
    ("10. Fine-tuned chart CNN", g("FT_RESULT")),
    ("11. Calibration temperatures", g("TEMPS", lambda T: {h: float(t) for h, t in T.items()})),
    ("12. Mean log loss over six horizons", g("mean_ll")),
    ("12. Every metric, every model, every horizon (valid and test)", g("TABLE")),
    ("13. v1.1 on test: gain over prior, placebo, rank IC, Holm", g("ST")),
    ("13A. Text coverage by year", g("TXT_COV")),
    ("13A. Seeds and epochs per configuration", g("SC_SEEDS")),
    ("13A. Modality scorecard, test, all rows", g("SC_TEST")),
    ("13A. Modality scorecard, test, rows with text", g("SC_TEST_TEXT")),
    ("13A. Modality scorecard, validation, all rows", g("SC_VALID")),
    ("13A. Modality scorecard, validation, rows with text", g("SC_VALID_TEXT")),
    ("13A. Quantile coverage with and without text", g("SC_COVER")),
    ("13A. What each input adds", g("SC_INC")),
    ("13A. Gain over the prior, year by year", g("SC_YEAR")),
    ("13A. Verdict", g("SC_VERDICT")),
    ("13B. E30: do the published fundamental scores rank returns? (pre-registered, Holm over six)", g("E30_TABLE")),
    ("13B. E30: parity with the documented result", g("E30_PARITY")),
    ("14. Mean log-loss gain over the prior on test (every model and saved run)", g("gain")),
    ("14. Reproduction of the documented E03 numbers", g("REPRO")),
    ("15. Gate 3: direction skill on validation vs test (replication, D-057)", g("SKT")),
    ("16. Documented backtest, test", g("BT_TEST")),
    ("16. Benchmark and PBO", g("BT_JSON", lambda b_: {"benchmark (equal-weight survivors), test":
                                                       b_.get("benchmark_ew_survivors", {}).get("test"), "PBO": b_.get("pbo")})),
    ("16A. Data checks", g("TRD_CHECKS")),
    ("16A. One bracket rule on every model: test (primary bracket)", g("TRD_TEST")),
    ("16A. One bracket rule on every model: validation (primary bracket; v1.1 omitted, in-sample)", g("TRD_VALID")),
    ("16A. Sensitivity: horizon-scaled bracket (sqrt-time)", g("TRD_SCALED")),
    ("16A. Stop-loss diagnostics", g("TRD_STOP")),
    ("17. Knowledge base: documented retrieval evaluation",
     g("E", lambda e: pd.DataFrame(e["book"]).T[["recall@1", "recall@5", "recall@10", "mrr", "ndcg@10"]])),
    ("18. Forward test chains", g("FWD")),
]
missing = [t for t, v in SECTIONS if v is None]
lines = [f"# NEXUS Alpha: end-to-end notebook report ({STAMP})", "",
         "Generated by the last cell of `notebooks/NEXUS_Alpha_End_to_End.ipynb`. Research only; nothing here is advice.", ""]
for title, v in SECTIONS:
    if v is None:
        continue
    lines += [f"## {title}", ""]
    lines.append(v if isinstance(v, str) else md_table(v))
    lines.append("")
if missing:
    lines += ["## Sections that did not run", "", *[f"* {m}" for m in missing], ""]
REPORT_MD = REPORT_DIR / f"NOTEBOOK_REPORT_{STAMP}.md"
REPORT_JSON = REPORT_DIR / f"NOTEBOOK_REPORT_{STAMP}.json"
REPORT_MD.write_text("\n".join(lines), encoding="utf-8")
REPORT_JSON.write_text(json.dumps({t: jsonable(v) for t, v in SECTIONS if v is not None} | {"missing": missing},
                                  indent=1, default=str), encoding="utf-8")
print(f"report written: {REPORT_MD}\n                {REPORT_JSON}")
print(f"{len(SECTIONS) - len(missing)} sections, {len(missing)} missing: {missing}")